# 实验七 · Softmax —— 规约、数值稳定性与算子融合的综合运用

**所属**：《并行计算》第六章 · 昇腾 Ascend C 算子开发　|　**难度**：⭐⭐⭐⭐⭐ 综合　|　**预计时长**：60–70 分钟

Softmax 是本章的综合实训，是因为它同时触及前面几个实验分头讨论过的三件事：**逐行的两次规约**（实验三）、**指数运算的量程**（实验四）、**中间张量的往返**（实验五）。

本实验回到 `bisheng` 核直调：一个 `.asc` 文件、一条编译命令。前四个版本依次处理这三件事并加上多核切分，第五个版本改用 CANN 的 `SoftMax` 高阶 API 实现同一件事。**它不引入新的算子。**

> **实验说明**
> 1. 本实验采用递进式的版本组织：每个版本只引入一个新的处理，结果表相应增加一行。
> 2. 请自上而下依次执行各单元格（Shift+Enter）。
> 3. 本实验依赖 **CANN 9.0.0 及以上**与 **Atlas A2/A3 训练推理系列产品**。
> 4. 本实验的主规格是 $2048 \times 1024$ 的 `float32` 矩阵，逐行做 Softmax。
> 5. 全部源代码由 `%%writefile` 写入 `src_softmax/`，再由一条 `bisheng` 命令编译。
> 6. 本实验的校验分三项：与真值的绝对误差、输出是否含 `inf` 与 `nan`、以及 Softmax 特有的**逐行求和为 1**。第 3 节说明为什么需要三项。

## 🎯 学习目标

完成本实验后，学生应能够：

- 说明 Softmax 的三趟数据依赖，指出它为何不能用一趟逐元素运算完成
- 说明减去行最大值为何不改变数学结果，却能消除上溢；并指出它不能消除下溢
- 用 `ReduceMax` 与 `ReduceSum` 完成逐行规约，说明它与实验三的核内规约是同一件事
- 用 `GetValue` 配合 `Adds` / `Muls` 完成标量与整行的运算，说明为何不需要广播接口
- 说明沿行切分为何没有核间合并，以及这一点如何决定它的扩展性
- 核算 Softmax 在融合前后的访存量，并与实测加速比对照
- 说明整行驻留片上缓冲是全融合的前提，并由片上容量定出行长上界
- 完成沿行方向的多核切分，处理行数不能被核数整除的情形
- 设计不变量校验，说明它相对于逐元素比对的独立价值
- 用 `AscendC::SoftMax` 高阶 API 实现同一个算子，对照官方算法框图指出它与手写版的结构性差别
- 说明高阶 API 对主机侧的两项依赖——切分参数从哪里来、临时空间由谁管理

## 🗺️ 学习路径

1. **准备阶段**：分析三趟依赖，核算融合前后的访存量，确定校验方案
2. **v1 · 非融合、不减最大值**：测量 `exp` 在 `float32` 上的溢出边界
3. **v2 · 非融合、减最大值**：引入 `ReduceMax` 与标量作用于整行的写法，量化数值稳定化的代价
4. **v3 · 全融合**：整行驻留片上缓冲，三趟都在片上完成
5. **v4 · 多核切行**：沿行方向切分，并验证行数不能被核数整除时的处理
6. **量程扫描**：沿输入幅度扫描，定出三项判据各自失效的位置
7. **v5 · 高阶 API**：用 `AscendC::SoftMax` 实现同一件事，与手写版对照
8. **分析与扩展**：行长上界与在线 Softmax

## 1. 背景与动机：三趟依赖

对矩阵 $X \in \mathbb{R}^{M \times N}$ 逐行做 Softmax：

$$ y_{ij} = \frac{e^{x_{ij}}}{\sum_{k=0}^{N-1} e^{x_{ik}}} $$

### 1.1 为什么不能一趟算完

前几个实验的逐元素算子满足 $z_i = f(x_i)$：读一个、算一个、写一个。Softmax 不满足这一性质——**分母依赖整行**。必须先把整行读完、算出分母，才能确定其中任何一个输出元素。

加上数值稳定化的处理，一共是**三趟**跨越整行的计算：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 趟次 | 计算 | 类型 |
| --- | --- | --- |
| 第一趟 | $m_i = \max_k x_{ik}$ | 规约 |
| 第二趟 | $t_{ij} = e^{x_{ij} - m_i}$，$s_i = \sum_k t_{ik}$ | 逐元素 + 规约 |
| 第三趟 | $y_{ij} = t_{ij} / s_i$ | 逐元素 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">趟次</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">计算</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">类型</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">第一趟</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">m_i = max_k x_ik</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">规约</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">第二趟</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">t_ij = e^x_ij - m_i，s_i = Σ_k t_ik</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">逐元素 + 规约</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">第三趟</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">y_ij = t_ij / s_i</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">逐元素</td>
</tr>
</tbody>
</table>

**这三趟的存在决定了本实验的全部设计。** 每一趟若各自成为一个核函数，中间张量就要往返 Global Memory；若整行能驻留片上缓冲，三趟就都在片上完成。

<img src="./images/06.07_three_pass.png" alt="06.07_three_pass"  width="960px" >

### 1.2 数值稳定性：为什么要减去行最大值

利用指数函数的性质，对任意常数 $c$：

$$ \frac{e^{x_{ij}}}{\sum_k e^{x_{ik}}} = \frac{e^{-c}\,e^{x_{ij}}}{e^{-c}\sum_k e^{x_{ik}}} = \frac{e^{x_{ij}-c}}{\sum_k e^{x_{ik}-c}} $$

**分子分母同乘 $e^{-c}$，结果不变。** 取 $c = m_i$（该行的最大值）之后，指数的自变量恒不大于 0，因而 $e^{x-m}$ 落在 $(0, 1]$ 之内，不会上溢。

`float32` 的表示上界约为 $3.4 \times 10^{38}$，对应 $e^{x}$ 在 $x > 88.7$ 时上溢。

**但 88.7 是单项 $e^{x}$ 的阈值，不是 Softmax 的阈值。** 分母是 $N$ 项之和，量级比最大的单项高出若干倍，因此**分母的溢出比单项更早发生**：当 $\max|x|$ 尚在 88.7 以下时，每个 $t_{ij}$ 都还是有限值，$s_i$ 却可能已经是 `inf`。第 11 节的量程扫描把这两个阈值分开呈现。

<img src="./images/06.07_stability.png" alt="06.07_stability"  width="980px" >

> **这一处理消除的是上溢，不是下溢。** 减去最大值后 $x - m$ 可能低至 $-200$，$e^{-200}$ 会下溢为 0。但这是**正确的**：真值本就小于 `float32` 能表示的最小数。这与实验四区分的两类失效是同一回事——**下溢丢掉的是本来就无法表示的信息，上溢破坏的是本来可以正确得到的结果。**

### 1.3 访存量核算

主规格取 $M = 2048$ 行、$N = 1024$ 列的 `float32`，单个张量 $2^{21}$ 个元素、8 MiB。

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 版本 | 各趟的访存 | 总访存 | 相对 v3 |
| --- | --- | --- | --- |
| v1 非融合、不减最大值 | 第二趟读 $x$ 写 $t$，第三趟读 $t$ 写 $y$ | $4 \times 8 = 32$ MiB | 2.00 |
| v2 非融合、减最大值 | 第一趟多读一次 $x$ | $5 \times 8 = 40$ MiB | 2.50 |
| v3 / v4 / v5 全融合 | 读 $x$ 写 $y$ | $2 \times 8 = 16$ MiB | 1.00 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">版本</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">各趟的访存</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">总访存</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">相对 v3</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v1 非融合、不减最大值</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">第二趟读 x 写 t，第三趟读 t 写 y</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">4 × 8 = 32 MiB</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">2.00</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v2 非融合、减最大值</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">第一趟多读一次 x</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">5 × 8 = 40 MiB</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">2.50</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v3 / v4 / v5 全融合</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">读 x 写 y</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">2 × 8 = 16 MiB</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">1.00</td>
</tr>
</tbody>
</table>

（行最大值与行和两个向量各只有 $M \times 4 = 8$ KiB，相对 8 MiB 可以忽略。）

**$2.50$ 就是 v2 到 v3 的理论加速比上界**，与实验五采用同一套分析方法。

## 2. 三处实现要点

### 2.1 标量与整行的运算：不需要广播接口

减去行最大值与除以行和，都是一个标量作用于整行。分两步完成：

```cpp
AscendC::ReduceMax(dstLocal, src, tmpLocal, cols);              // 结果写入 dstLocal[0]
LabDType maxVal = dstLocal.GetValue(0);                         // Scalar 流水取出标量
AscendC::Adds(dst, src, static_cast<LabDType>(-maxVal), cols);  // 标量作用于整行
```

`Adds` 与 `Muls` 的第三个参数本就是标量，**因此无须使用任何广播类接口**。

**但 `GetValue` 不是免费的。** 各计算单元有各自独立的指令队列，并行执行；队列之间靠同步指令衔接：

<img src="./images/06.07_vector_instr_queues.png" alt="06.07_vector_instr_queues" width="900px">

*Vector 编程范式指令队列示例。搬入、计算、搬出三个单元各有一条指令队列，`EnQue` 发射同步指令 set、`DeQue` 发射同步指令 wait*

标量读写落在其中的另一条流水上，官方对此写得很清楚：

> Scalar 读写 Global Memory 和 Unified Buffer 时属于 PIPE_S（Scalar 流水）操作，当用户使用 SetValue 或者 GetValue 接口，且算子工程使能自动同步时，不需要手动插入同步事件。
> ——《Ascend C 算子开发指南》 Scalar 读写数据时的同步

官方给出的手工同步示例中，`GetValue` 之后紧跟 `SetFlag<HardEvent::S_V>`，注释写明「因此 Vector 流水需要等待 Scalar 操作结束」。**自动同步免去的是手工插入事件，不是等待本身**：本实验每行取两次标量，就是两次矢量流水的等待。第 13 节 ⑤ 会用到这一点。

### 2.2 用乘法代替除法

第三趟是 $y = t / s$。逐元素做除法需要 $N$ 次除法；改为先算标量倒数、再做乘法，每行只需 1 次除法：

```cpp
AscendC::Muls(dst, src, static_cast<LabDType>(1) / sumVal, cols);
```

**除法在矢量单元上的代价高于乘法**，这一改写在长行上收益可观。同一条思路在实验五用 `Reciprocal` 替代逐元素除法时已经用过。

### 2.3 行驻留：全融合的前提，也是它的限制

全融合要求整行驻留片上缓冲。以全融合核函数为例，片上占用由四项构成：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 项 | 大小 | 说明 |
| --- | --- | --- |
| 输入队列 | N × 4 × 2 | 队列深度为 2 |
| 输出队列 | N × 4 × 2 | 队列深度为 2 |
| 规约临时空间 | TMP × 4 ≈ N/2 | <code>ReduceMax</code> 与 <code>ReduceSum</code> 共用 |
| 规约结果 | 32 字节 | 只存一个标量，但搬运以 32 字节为单位 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">项</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">大小</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">说明</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">输入队列</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">N × 4 × 2</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">队列深度为 2</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">输出队列</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">N × 4 × 2</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">队列深度为 2</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">规约临时空间</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">TMP × 4 ≈ N/2</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>ReduceMax</code> 与 <code>ReduceSum</code> 共用</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">规约结果</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">32 字节</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">只存一个标量，但搬运以 32 字节为单位</td>
</tr>
</tbody>
</table>

四项之和不得超过 UB 容量，由此定出行长的上界：

$$ 16N + \text{TMP} \times 4 + 32 \;\le\; |UB| $$

UB 的容量由硬件规格给出。官方对本实验所用的架构版本写明：

> 针对 NPU 架构版本 220x：UB 总大小为 192KB，包含 16 个 bank group，每个 bank group 包含 3 个 bank。每个 bank 大小为 4KB，由 128 行组成，每行长度为 32B。
> ——《Ascend C 算子开发指南》 Unified Buffer 的 bank 结构

<img src="./images/06.07_ub_bank_220x.png" alt="06.07_ub_bank_220x" width="1000px">

*bank 结构示意图（NPU 架构版本 220x）。16 个 bank group × 3 个 bank × 4KB 即 192KB；每行 32B 也正是本章反复出现的搬运粒度。另一类产品的 UB 为 256KB，核算时应按实际型号取值*

本实验编译时指定的 `dav-2201` 即属于 220x，以 $|UB| = 192$ KiB 计，全融合核函数的行长上界约为 $1.2 \times 10^{4}$。

**但全融合核函数并不是最先超限的那一个。** 把同一条式子分别套到六个核函数上，
得到的上界相差两倍以上：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 核函数 | 片上占用（字节，R = ROWBUF_MAX） | 行长上界 |
| --- | --- | --- |
| <code>softmax_api</code>（v5） | 8N + 8N + 8N（API 临时空间按一行的两倍预留） | ≈ 8.2 × 10³ |
| <code>softmax_expsum_submax</code>（v2） | 16N + N/2 + 8R + 32（rowBuf 与 mBuf 两块行标量缓冲） | ≈ 9.9 × 10³ |
| <code>softmax_expsum_nomax</code>（v1） | 16N + N/2 + 4R + 32（不减最大值，mBuf 被编译期裁掉） | ≈ 1.09 × 10⁴ |
| <code>softmax_divide</code>（v1、v2） | 16N + 4R | ≈ 1.13 × 10⁴ |
| <code>softmax_fused</code>（v3、v4） | 16N + N/2 + 32 | ≈ 1.19 × 10⁴ |
| <code>softmax_rowmax</code>（v2） | 8N + N/2 + 4R + 32（只有输入队列） | ≈ 2.1 × 10⁴ |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">核函数</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">片上占用（字节，R = ROWBUF_MAX）</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">行长上界</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>softmax_api</code>（v5）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">8N + 8N + 8N（API 临时空间按一行的两倍预留）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">≈ 8.2 × 10³</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>softmax_expsum_submax</code>（v2）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">16N + N/2 + 8R + 32（rowBuf 与 mBuf 两块行标量缓冲）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">≈ 9.9 × 10³</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>softmax_expsum_nomax</code>（v1）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">16N + N/2 + 4R + 32（不减最大值，mBuf 被编译期裁掉）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">≈ 1.09 × 10⁴</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>softmax_divide</code>（v1、v2）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">16N + 4R</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">≈ 1.13 × 10⁴</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>softmax_fused</code>（v3、v4）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">16N + N/2 + 32</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">≈ 1.19 × 10⁴</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>softmax_rowmax</code>（v2）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">8N + N/2 + 4R + 32（只有输入队列）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">≈ 2.1 × 10⁴</td>
</tr>
</tbody>
</table>

先超限的是 v5 的 `softmax_api`：它为高阶 API 预留的临时空间本身就随行长增长
（第 7.6 节）。其次是 v2 用的 `softmax_expsum_submax`，它比全融合核函数多两块行标量缓冲；
v1 用的 `softmax_expsum_nomax` 少一块（`mBuf` 由 `if constexpr` 在编译期裁掉），
上界因此高出一截。
**六个核函数编译在同一个文件里，因此整个文件能编过的行长，取决于其中上界最低的那一个。**
动手练习第 1 题据此设计。

<img src="./images/06.07_row_resident.png" alt="06.07_row_resident"  width="1000px" >

**行长超过上界时，全融合就不再可行**：整行装不进片上，三趟无法串起来。
此时要么退回非融合，要么改用边扫边修正的在线算法——动手练习第 5 题给出它的递推式。


## 3. 三项判据

前几个实验只用一条绝对误差判据。Softmax 需要三项，理由在于**它的失效方式不止一种**。

### 3.1 逐元素比对

真值由 `double` 精度的参考实现给出。容差沿用实验二至实验五那条公式：Softmax 的输出上界为 1，累加链长度为 $N$，因此

$$ \text{atol} = c \cdot \varepsilon \cdot \sqrt{N}, \qquad c = 2 $$

### 3.2 输出是否含 `inf` 与 `nan`

上溢产生 `inf`，`inf / inf` 产生 `nan`。**这两者与真值的差是 `nan`，任何基于比较的判据都不会为真**——因此必须单独检查，不能指望绝对误差把它捕捉出来。

### 3.3 不变量：逐行求和为 1

$\sum_k y_{ik} = 1$ 是 Softmax 的定义所蕴含的，与真值从何而来无关。**它是一条与逐元素比对相互独立的判据**：不需要参考实现，只需要输出本身。

行和是 $N$ 个 `float32` 输出在 `double` 中累加，误差量级约为 $\varepsilon\sqrt{N}$。本实验取 $10^{-4}$，比这个量级宽两个数量级——**它要捕捉的是结构性的失效，不是舍入。**

三项并列，不是为了保险，而是因为它们捕捉到的失效并不相同。第 11 节据此设计了量程扫描。

## 4. 本实验的测量方法

`cpu_ms` 与 `kernel_ms` 两个口径与前几个实验一致，计时三纪律同样不再复述。两者分别落在两类记录行上：`cpu_ms` 在 `[BASE]` 行，`kernel_ms` 在 `[PERF]` 行。

**本实验不做端到端计时。** 这与实验二起确定的口径一致：加速比一律以核函数耗时为准，主机与设备之间的搬运只在讨论卸载收益时才计入，那一部分放在实验五。

两处需要说明：

**CPU 基准采用与 v3 相同的算法**——`float` 精度、全融合、逐行处理、单线程。加速比以它为分母，因此加速比反映的是同一个算法在两种处理器上的差距，不含算法本身的差别。

**v1 与 v2 之间只差一趟。** 两者都是非融合，唯一的差别是 v2 多一趟 `ReduceMax` 并在第二趟多一条 `Adds`。因此 $t_{v2} - t_{v1}$ 就是**数值稳定化的时间代价**，可以直接读出。

## 5. 环境准备与检查

In [ ]:
!mkdir -p src_softmax

import os, subprocess

env = subprocess.check_output(
    "bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True
)
for line in env.splitlines():
    if "=" in line:
        os.environ.__setitem__(*line.split("=", 1))
print("🎉 环境变量导入完成")

In [ ]:
import shutil, subprocess

print("bisheng  :", shutil.which("bisheng") or "⚠️  未找到，请重新执行上一个单元格")
print("npu-smi  :", shutil.which("npu-smi") or "⚠️  未找到")
if shutil.which("npu-smi"):
    print()
    print(
        subprocess.run(["npu-smi", "info"], capture_output=True, text=True).stdout[
            :1800
        ]
    )

## 6. 版本设计总览

五个版本每次只改一处，因此每一行的差值都能归因到一个具体的改动上。

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 版本 | 组织方式 | 减最大值 | 核数 | 访存量 | 相对上一版新增的处理 |
| --- | --- | --- | --- | --- | --- |
| **v1** | 非融合，两个核函数 | 否 | 1 | 32 MiB | 基准：逐行规约、行标量攒够再写 |
| **v2** | 非融合，三个核函数 | **是** | 1 | 40 MiB | `ReduceMax` 一趟 + 标量作用于整行 |
| **v3** | **全融合**，一个核函数 | 是 | 1 | **16 MiB** | 整行驻留片上，三趟不落回 Global Memory |
| **v4** | 全融合 | 是 | **8** | 16 MiB | 沿行方向切分到多个核 |
| **v5** | 全融合，**改用高阶 API** | 是 | 8 | 16 MiB | 三趟交给 `AscendC::SoftMax` 完成 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">版本</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">组织方式</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">减最大值</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">核数</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">访存量</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">相对上一版新增的处理</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>v1</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">非融合，两个核函数</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">否</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">1</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">32 MiB</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">基准：逐行规约、行标量攒够再写</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>v2</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">非融合，三个核函数</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>是</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">1</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">40 MiB</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>ReduceMax</code> 一趟 + 标量作用于整行</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>v3</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>全融合</strong>，一个核函数</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">是</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">1</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>16 MiB</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">整行驻留片上，三趟不落回 Global Memory</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>v4</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">全融合</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">是</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>8</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">16 MiB</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">沿行方向切分到多个核</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>v5</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">全融合，<strong>改用高阶 API</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">是</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">8</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">16 MiB</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">三趟交给 <code>AscendC::SoftMax</code> 完成</td>
</tr>
</tbody>
</table>

四处对照各自回答一个问题：

- **v1 → v2**：数值稳定化的时间代价；
- **v2 → v3**：融合省下的访存折算为多少耗时，与第 1.3 节的 $2.50$ 对照；
- **v3 → v4**：沿行切分的收益。两者是同一个核函数，差别只在核数；因为没有核间合并，比值应当接近核数之比；
- **v4 → v5**：手写实现与高阶 API 的差距。两者核数相同、访存量相同、切分方式相同，**差别只在三趟由谁完成**。

v1 不做数值稳定化处理，因此它的可用范围由输入幅度与 `float32` 的量程共同决定，**第 11 节沿输入幅度扫描正是用来定出这个范围的**。无论校验是否通过，它与 v2 的算法差别只有一趟，两者的耗时可以直接相减，因此仍可用作计时基准。

## 7. Device 侧实现

全部源码写入同一个 `.asc` 文件，分段追加，顺序即编译顺序。五个版本共用六个核函数：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 核函数 | 由谁使用 | 作用 |
| --- | --- | --- |
| `softmax_rowmax` | v2 | 第一趟：逐行 `ReduceMax` |
| `softmax_expsum_nomax` | v1 | 第二趟：$t = e^{x}$ 并求行和 |
| `softmax_expsum_submax` | v2 | 第二趟：$t = e^{x-m}$ 并求行和 |
| `softmax_divide` | v1、v2 | 第三趟：$y = t \cdot (1/s)$ |
| `softmax_fused` | v3、v4 | 三趟合并，整行驻留片上 |
| `softmax_api` | v5 | 三趟交给 `AscendC::SoftMax` 高阶 API |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">核函数</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">由谁使用</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">作用</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>softmax_rowmax</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v2</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">第一趟：逐行 <code>ReduceMax</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>softmax_expsum_nomax</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v1</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">第二趟：t = e^x 并求行和</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>softmax_expsum_submax</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v2</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">第二趟：t = e^x-m 并求行和</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>softmax_divide</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v1、v2</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">第三趟：y = t · (1/s)</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>softmax_fused</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v3、v4</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">三趟合并，整行驻留片上</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>softmax_api</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v5</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">三趟交给 <code>AscendC::SoftMax</code> 高阶 API</td>
</tr>
</tbody>
</table>

六个核函数共用同一组头文件：设备侧只需 `kernel_operator.h`，v5 用到的 `AscendC::SoftMax` 也由它引入，无须再单独包含 activation 相关的头文件。

### 7.1 文件头与参数

可调参数集中在文件开头，写成可由 `-D` 覆盖的形式，改动核数或列数时在这里重新编译即可；其余由它们推导出来的量写成 `constexpr`，改一处即可全部跟着变。

三个量值得单独说明：

- **`TMP_LENGTH`** 是 `ReduceMax` 与 `ReduceSum` 需要的临时空间，按实验三第 2.1 节那条核算式写成表达式，改动列数无须手工重算。
- **`ROWBUF_MAX`** 是行标量缓冲的容量，只有前三个核函数用到——每行产出一个标量，先攒在片上，最后一次性搬出（第 7.2 节）。**它按 `ROWS_MAX` 预留**，因为这三个核函数在本实验中都以单核下发，一个核要容纳全部行的标量。全融合核函数不产出行标量，没有这块缓冲。
- **`ROWS_MAX`** 是行数的编译期上界，主程序在开头检查，超出即报错退出。它与 `LAB_ROWS` 不是一回事：后者只是运行时的默认行数，前者才是缓冲的依据。

`API_TMP_BYTES` 是留给 v5 的临时空间，第 7.6 节说明它的作用。

In [ ]:
%%writefile src_softmax/ascendc_softmax.asc
/**
 * 并行计算 第六章 实验七：Softmax
 *
 * 本文件包含五个版本的核函数、CPU 基准、数据生成、精度校验与 main，
 * 由一条 bisheng 命令编译为单个可执行程序：
 *
 *   bisheng src_softmax/ascendc_softmax.asc --npu-arch=dav-2201 -O2 -o src_softmax/ascendc_softmax
 *
 * 用法：
 *   ./ascendc_softmax                     主规格，五个版本对照
 *   ./ascendc_softmax <rows> <cols>       指定矩阵形状
 *   ./ascendc_softmax <rows> <cols> <amp> 指定输入幅度（量程扫描用）
 */
#include <cstdio>
#include <cstdint>
#include <cstdlib>
#include <cmath>
#include <ctime>
#include <vector>

#include "acl/acl.h"           // Host 侧
#include "kernel_operator.h"   // Device 侧

/* ===================== 可由命令行 -D 覆盖的参数 ===================== */

using LabDType = float; /* 全程 float32，与实验二至实验五的精度校验口径一致 */

/* 行数的默认值与上界。默认值可由命令行覆盖，上界决定片上缓冲的大小 */
#ifndef LAB_ROWS
#define LAB_ROWS (2048)
#endif
constexpr uint32_t ROWS_DEFAULT = static_cast<uint32_t>(LAB_ROWS);

#ifndef LAB_ROWS_MAX
#define LAB_ROWS_MAX (4096)
#endif
constexpr uint32_t ROWS_MAX = static_cast<uint32_t>(LAB_ROWS_MAX);

#ifndef LAB_COLS
#define LAB_COLS (1024)
#endif
constexpr uint32_t COLS_MAX = static_cast<uint32_t>(LAB_COLS);

/* 参与计算的核数（v4 使用；v1、v2、v3 固定为 1） */
#ifndef LAB_BLOCK_DIM
#define LAB_BLOCK_DIM (8)
#endif
constexpr uint32_t BLOCK_DIM = static_cast<uint32_t>(LAB_BLOCK_DIM);

/* 计时参数：预热次数与重复次数 */
#ifndef LAB_REPEAT
#define LAB_REPEAT (20)
#endif
constexpr int32_t REPEAT = static_cast<int32_t>(LAB_REPEAT);
constexpr int32_t WARMUP = 3;

/* 输入幅度的默认值：x 取 [-AMP, AMP] 上的均匀分布 */
#ifndef LAB_AMP
#define LAB_AMP (100)
#endif

/* 对齐常量：搬运以 32 字节为单位，对 float 即 8 个元素 */
constexpr uint32_t ALIGN_ELEM = 32 / sizeof(LabDType);

/* 队列深度：TQue 模板的第二个参数，编译期常量 */
constexpr uint32_t QUEUE_DEPTH = 2;

/* 规约临时缓冲的元素数，与实验三第 2.1 节同一条核算式：
 *   elementsPerRepeat = 256 / sizeof(float) = 64
 *   firstMaxRepeat    = COLS / 64
 *   tmpElements       = RoundUp(firstMaxRepeat, 8) * 8
 * COLS = 1024 时算得 128 */
constexpr uint32_t TMP_LENGTH = ((COLS_MAX / 64 + 7) / 8) * 8 * 8;

/* 行标量缓冲的容量。**必须按单核处理全部行预留**：v1、v2、v3 都以单核运行，
 * 此时一个核要攒下所有行的标量。按每核分到的行数预留会在单核时越界。 */
constexpr uint32_t ROWBUF_MAX = ROWS_MAX + ALIGN_ELEM;

/* v5 交给 SoftMax 高阶 API 的临时空间。官方的做法是由主机侧的
 * GetSoftMaxMinTmpSize 与 GetSoftMaxMaxTmpSize 定出上下界，再在其中选一个值：
 * 最小空间保证功能正确，最大空间用于提升性能，见 7.6 节。
 * 本实验按一行的两倍预留，可以让 API 把一行一次算完 */
constexpr uint32_t API_TMP_BYTES = COLS_MAX * sizeof(LabDType) * 2;

### 7.2 第一趟：行最大值

每行调用一次 `ReduceMax`，把结果以标量形式存入片上缓冲；本核负责的全部行处理完毕后，一次性搬出。

**为什么要先攒后写**：规约结果每行只有一个 `float`，逐行写出即每次只写 4 字节，**违反 32 字节的搬运粒度**。实验三处理规约输出时用的是同一条约束——那里只有一个标量，补到 8 个元素即可；这里有多行标量，因此在片上攒够再整体搬出。

搬出长度向上取整到 32 字节的整数倍，尾部补 0。

**补位的元素落在本核区间之后**，因此这段代码只在单核下发时成立：若改为多核，且每个核分到的行数不是 8 的倍数，前一个核补出的 0 会覆盖后一个核区间开头的真实行标量。本实验的 v1、v2 固定以单核下发，因此不受影响；**若要把这两个核函数改成多核，必须先处理这一点**——办法是给每个核一段独立对齐的输出区间。

In [ ]:
%%writefile -a src_softmax/ascendc_softmax.asc
/* ===================== v1、v2 第一趟：行最大值 =====================
 * 新增概念：ReduceMax，以及行标量攒够再写
 *   每行调用一次 ReduceMax 得到该行的最大值，以标量形式存入片上缓冲；
 *   本核负责的全部行处理完毕后，一次性把这段行标量搬出到 Global Memory。
 * 逐行写出只有 4 字节，违反 32 字节的搬运粒度，因此必须先攒后写。
 */
class KernelRowMax {
 public:
  __aicore__ inline KernelRowMax() {}

  __aicore__ inline void Init(GM_ADDR x, GM_ADDR m, uint32_t cols,
                              uint32_t rowBegin, uint32_t rowCount) {
    cols_ = cols;
    rowBegin_ = rowBegin;
    rowCount_ = rowCount;
    xGm.SetGlobalBuffer(reinterpret_cast<__gm__ LabDType *>(x), 0);
    mGm.SetGlobalBuffer(reinterpret_cast<__gm__ LabDType *>(m), 0);

    pipe.InitBuffer(inQueue, QUEUE_DEPTH, COLS_MAX * sizeof(LabDType));
    pipe.InitBuffer(rowBuf, ROWBUF_MAX * sizeof(LabDType));
    pipe.InitBuffer(tmpBuf, TMP_LENGTH * sizeof(LabDType));
    pipe.InitBuffer(dstBuf, ALIGN_ELEM * sizeof(LabDType));
  }

  __aicore__ inline void Process() {
    AscendC::LocalTensor<LabDType> rowLocal = rowBuf.Get<LabDType>();
    AscendC::LocalTensor<LabDType> tmpLocal = tmpBuf.Get<LabDType>();
    AscendC::LocalTensor<LabDType> dstLocal = dstBuf.Get<LabDType>();

    for (uint32_t r = 0; r < rowCount_; ++r) {
      AscendC::LocalTensor<LabDType> xLocal = inQueue.AllocTensor<LabDType>();
      AscendC::DataCopy(xLocal, xGm[(rowBegin_ + r) * cols_], cols_);
      inQueue.EnQue(xLocal);

      AscendC::LocalTensor<LabDType> src = inQueue.DeQue<LabDType>();
      AscendC::ReduceMax(dstLocal, src, tmpLocal, cols_);
      /* GetValue 属于 Scalar 流水，自动同步下无须手工插入同步事件 */
      rowLocal.SetValue(r, dstLocal.GetValue(0));
      inQueue.FreeTensor(src);
    }

    /* 攒够之后一次搬出。搬运长度向上取整到 32 字节 */
    uint32_t outLen = ((rowCount_ + ALIGN_ELEM - 1) / ALIGN_ELEM) * ALIGN_ELEM;
    for (uint32_t r = rowCount_; r < outLen; ++r) {
      rowLocal.SetValue(r, static_cast<LabDType>(0));
    }
    AscendC::DataCopy(mGm[rowBegin_], rowLocal, outLen);
  }

 private:
  AscendC::TPipe pipe;
  AscendC::TQue<AscendC::TPosition::VECIN, QUEUE_DEPTH> inQueue;
  AscendC::TBuf<AscendC::TPosition::VECCALC> rowBuf;
  AscendC::TBuf<AscendC::TPosition::VECCALC> tmpBuf;
  AscendC::TBuf<AscendC::TPosition::VECCALC> dstBuf;
  AscendC::GlobalTensor<LabDType> xGm;
  AscendC::GlobalTensor<LabDType> mGm;
  uint32_t cols_ = 0, rowBegin_ = 0, rowCount_ = 0;
};

### 7.3 第二趟：取指数并求行和

模板参数 `SUB_MAX` 决定是否先减去行最大值：`false` 即 v1，`true` 即 v2。**两个版本只差一条 `Adds` 与一次读入 $m$**，因此用同一段代码表达，由编译期常量裁剪——这与实验五用模板参数区分算子链的做法一致，运行时没有判断开销。

五处细节：

- 行最大值向量**整段读入**，之后逐行用 `GetValue` 取标量。若逐行从 Global Memory 读一个 `float`，同样违反搬运粒度。
- `Adds` 的第三个参数是标量，直接作用于整行，**这就是第 2.1 节所说的无须广播接口**。
- 求和用的是 `ReduceSum`，与实验三同一个接口；输入是刚算完的 $t$，因此这一趟同时产出 $t$ 与 $s$。
- 行和向量的补位取 **1** 而不是 0（第一趟的行最大值补 0）。**两处补位的取值按各自被使用的方式选定**：行和会成为分母，取 1 可保证即便被误读也不会产生除零。
- 行最大值缓冲 `mBuf` 也放在 `if constexpr (SUB_MAX)` 之内。**编译期裁掉的不只是指令，还有片上空间**：v1 用的那份实例少一块行标量缓冲，行长上界因此比 v2 的那份高出一截（第 2.3 节）。

In [ ]:
%%writefile -a src_softmax/ascendc_softmax.asc
/* ===================== v1、v2 第二趟：取指数并求行和 =====================
 * 模板参数 SUB_MAX 决定是否先减去行最大值：
 *   false —— t = exp(x)，即 v1；
 *   true  —— t = exp(x - m)，即 v2。
 * 两者只差一条 Adds 与一次读入 m，因此可以用同一段代码表达，
 * 版本之间的差别由编译期常量裁剪，运行时没有判断开销。
 */
template <bool SUB_MAX>
class KernelExpSum {
 public:
  __aicore__ inline KernelExpSum() {}

  __aicore__ inline void Init(GM_ADDR x, GM_ADDR m, GM_ADDR t, GM_ADDR s,
                              uint32_t cols, uint32_t rowBegin,
                              uint32_t rowCount) {
    cols_ = cols;
    rowBegin_ = rowBegin;
    rowCount_ = rowCount;
    xGm.SetGlobalBuffer(reinterpret_cast<__gm__ LabDType *>(x), 0);
    mGm.SetGlobalBuffer(reinterpret_cast<__gm__ LabDType *>(m), 0);
    tGm.SetGlobalBuffer(reinterpret_cast<__gm__ LabDType *>(t), 0);
    sGm.SetGlobalBuffer(reinterpret_cast<__gm__ LabDType *>(s), 0);

    pipe.InitBuffer(inQueue, QUEUE_DEPTH, COLS_MAX * sizeof(LabDType));
    pipe.InitBuffer(outQueue, QUEUE_DEPTH, COLS_MAX * sizeof(LabDType));
    pipe.InitBuffer(rowBuf, ROWBUF_MAX * sizeof(LabDType));
    if constexpr (SUB_MAX) {
      /* 行最大值缓冲只有 v2 用得到。v1 不减最大值，这块空间在编译期就被裁掉，
       * 它的片上占用因此比 v2 少一块行标量缓冲——见 2.3 节的上界表 */
      pipe.InitBuffer(mBuf, ROWBUF_MAX * sizeof(LabDType));
    }
    pipe.InitBuffer(tmpBuf, TMP_LENGTH * sizeof(LabDType));
    pipe.InitBuffer(dstBuf, ALIGN_ELEM * sizeof(LabDType));
  }

  __aicore__ inline void Process() {
    AscendC::LocalTensor<LabDType> rowLocal = rowBuf.Get<LabDType>();
    AscendC::LocalTensor<LabDType> tmpLocal = tmpBuf.Get<LabDType>();
    AscendC::LocalTensor<LabDType> dstLocal = dstBuf.Get<LabDType>();
    /* 先只声明，不取用：v1 没有分配 mBuf，取用会越界 */
    AscendC::LocalTensor<LabDType> mLocal;

    uint32_t rowLen = ((rowCount_ + ALIGN_ELEM - 1) / ALIGN_ELEM) * ALIGN_ELEM;
    if constexpr (SUB_MAX) {
      /* 行最大值向量整段读入，之后逐行取标量 */
      mLocal = mBuf.Get<LabDType>();
      AscendC::DataCopy(mLocal, mGm[rowBegin_], rowLen);
    }

    for (uint32_t r = 0; r < rowCount_; ++r) {
      AscendC::LocalTensor<LabDType> xLocal = inQueue.AllocTensor<LabDType>();
      AscendC::DataCopy(xLocal, xGm[(rowBegin_ + r) * cols_], cols_);
      inQueue.EnQue(xLocal);

      AscendC::LocalTensor<LabDType> src = inQueue.DeQue<LabDType>();
      AscendC::LocalTensor<LabDType> dst = outQueue.AllocTensor<LabDType>();
      if constexpr (SUB_MAX) {
        /* 标量作用于整行：Adds 的第三个参数本就是标量，无须广播接口 */
        AscendC::Adds(dst, src, static_cast<LabDType>(-mLocal.GetValue(r)),
                      cols_);
        AscendC::Exp(dst, dst, cols_);
      } else {
        AscendC::Exp(dst, src, cols_);
      }
      AscendC::ReduceSum(dstLocal, dst, tmpLocal, cols_);
      rowLocal.SetValue(r, dstLocal.GetValue(0));

      outQueue.EnQue(dst);
      inQueue.FreeTensor(src);

      AscendC::LocalTensor<LabDType> res = outQueue.DeQue<LabDType>();
      AscendC::DataCopy(tGm[(rowBegin_ + r) * cols_], res, cols_);
      outQueue.FreeTensor(res);
    }

    for (uint32_t r = rowCount_; r < rowLen; ++r) {
      rowLocal.SetValue(r, static_cast<LabDType>(1));
    }
    AscendC::DataCopy(sGm[rowBegin_], rowLocal, rowLen);
  }

 private:
  AscendC::TPipe pipe;
  AscendC::TQue<AscendC::TPosition::VECIN, QUEUE_DEPTH> inQueue;
  AscendC::TQue<AscendC::TPosition::VECOUT, QUEUE_DEPTH> outQueue;
  AscendC::TBuf<AscendC::TPosition::VECCALC> rowBuf;
  AscendC::TBuf<AscendC::TPosition::VECCALC> mBuf;
  AscendC::TBuf<AscendC::TPosition::VECCALC> tmpBuf;
  AscendC::TBuf<AscendC::TPosition::VECCALC> dstBuf;
  AscendC::GlobalTensor<LabDType> xGm, mGm, tGm, sGm;
  uint32_t cols_ = 0, rowBegin_ = 0, rowCount_ = 0;
};

### 7.4 第三趟：逐行归一化

行和向量整段读入，每行取出标量、求倒数、再用 `Muls` 作用于整行。**每行只做一次除法**，其余是乘法（第 2.2 节）。

In [ ]:
%%writefile -a src_softmax/ascendc_softmax.asc
/* ===================== v1、v2 第三趟：逐行归一化 =====================
 * y = t * (1 / s)。先算标量倒数、再做乘法，每行只做一次除法；
 * 若逐元素做除法，则要做 cols 次。
 */
class KernelDivide {
 public:
  __aicore__ inline KernelDivide() {}

  __aicore__ inline void Init(GM_ADDR t, GM_ADDR s, GM_ADDR y, uint32_t cols,
                              uint32_t rowBegin, uint32_t rowCount) {
    cols_ = cols;
    rowBegin_ = rowBegin;
    rowCount_ = rowCount;
    tGm.SetGlobalBuffer(reinterpret_cast<__gm__ LabDType *>(t), 0);
    sGm.SetGlobalBuffer(reinterpret_cast<__gm__ LabDType *>(s), 0);
    yGm.SetGlobalBuffer(reinterpret_cast<__gm__ LabDType *>(y), 0);

    pipe.InitBuffer(inQueue, QUEUE_DEPTH, COLS_MAX * sizeof(LabDType));
    pipe.InitBuffer(outQueue, QUEUE_DEPTH, COLS_MAX * sizeof(LabDType));
    pipe.InitBuffer(sBuf, ROWBUF_MAX * sizeof(LabDType));
  }

  __aicore__ inline void Process() {
    AscendC::LocalTensor<LabDType> sLocal = sBuf.Get<LabDType>();
    uint32_t rowLen = ((rowCount_ + ALIGN_ELEM - 1) / ALIGN_ELEM) * ALIGN_ELEM;
    AscendC::DataCopy(sLocal, sGm[rowBegin_], rowLen);

    for (uint32_t r = 0; r < rowCount_; ++r) {
      AscendC::LocalTensor<LabDType> tLocal = inQueue.AllocTensor<LabDType>();
      AscendC::DataCopy(tLocal, tGm[(rowBegin_ + r) * cols_], cols_);
      inQueue.EnQue(tLocal);

      AscendC::LocalTensor<LabDType> src = inQueue.DeQue<LabDType>();
      AscendC::LocalTensor<LabDType> dst = outQueue.AllocTensor<LabDType>();
      LabDType invSum = static_cast<LabDType>(1) / sLocal.GetValue(r);
      AscendC::Muls(dst, src, invSum, cols_);
      outQueue.EnQue(dst);
      inQueue.FreeTensor(src);

      AscendC::LocalTensor<LabDType> res = outQueue.DeQue<LabDType>();
      AscendC::DataCopy(yGm[(rowBegin_ + r) * cols_], res, cols_);
      outQueue.FreeTensor(res);
    }
  }

 private:
  AscendC::TPipe pipe;
  AscendC::TQue<AscendC::TPosition::VECIN, QUEUE_DEPTH> inQueue;
  AscendC::TQue<AscendC::TPosition::VECOUT, QUEUE_DEPTH> outQueue;
  AscendC::TBuf<AscendC::TPosition::VECCALC> sBuf;
  AscendC::GlobalTensor<LabDType> tGm, sGm, yGm;
  uint32_t cols_ = 0, rowBegin_ = 0, rowCount_ = 0;
};

### 7.5 全融合：v3 与 v4 共用

整行读入片上之后，三趟依次在片上完成，中间结果不落回 Global Memory。与前三个核函数相比，这里少掉的不是计算，而是 $t$ 的一次写出与一次读入、以及 $x$ 的一次重复读入。

注意 `dst` 被复用了三次：`Adds` 写入它、`Exp` 原地覆盖、`Muls` 再原地覆盖。**源与目的地址相同是允许的**，这一点在实验五第 2.3 节讨论过。

v3 与 v4 是同一个核函数，差别只在启动的核数——**因此 v3 到 v4 的差值只能归因于并行度**。

In [ ]:
%%writefile -a src_softmax/ascendc_softmax.asc
/* ===================== v3、v4：全融合 =====================
 * 一行读入片上之后，三趟全部在片上完成，中间结果不落回 Global Memory。
 * 访存量因此从读三次、写两次降到读一次、写一次。
 * v3 与 v4 是同一个核函数，差别只在启动的核数。
 */
class KernelSoftmaxFused {
 public:
  __aicore__ inline KernelSoftmaxFused() {}

  __aicore__ inline void Init(GM_ADDR x, GM_ADDR y, uint32_t cols,
                              uint32_t rowBegin, uint32_t rowCount) {
    cols_ = cols;
    rowBegin_ = rowBegin;
    rowCount_ = rowCount;
    xGm.SetGlobalBuffer(reinterpret_cast<__gm__ LabDType *>(x), 0);
    yGm.SetGlobalBuffer(reinterpret_cast<__gm__ LabDType *>(y), 0);

    pipe.InitBuffer(inQueue, QUEUE_DEPTH, COLS_MAX * sizeof(LabDType));
    pipe.InitBuffer(outQueue, QUEUE_DEPTH, COLS_MAX * sizeof(LabDType));
    pipe.InitBuffer(tmpBuf, TMP_LENGTH * sizeof(LabDType));
    pipe.InitBuffer(dstBuf, ALIGN_ELEM * sizeof(LabDType));
  }

  __aicore__ inline void Process() {
    AscendC::LocalTensor<LabDType> tmpLocal = tmpBuf.Get<LabDType>();
    AscendC::LocalTensor<LabDType> dstLocal = dstBuf.Get<LabDType>();

    for (uint32_t r = 0; r < rowCount_; ++r) {
      AscendC::LocalTensor<LabDType> xLocal = inQueue.AllocTensor<LabDType>();
      AscendC::DataCopy(xLocal, xGm[(rowBegin_ + r) * cols_], cols_);
      inQueue.EnQue(xLocal);

      AscendC::LocalTensor<LabDType> src = inQueue.DeQue<LabDType>();
      AscendC::LocalTensor<LabDType> dst = outQueue.AllocTensor<LabDType>();

      /* 第一趟：行最大值 */
      AscendC::ReduceMax(dstLocal, src, tmpLocal, cols_);
      LabDType maxVal = dstLocal.GetValue(0);

      /* 第二趟：减最大值、取指数、求和 */
      AscendC::Adds(dst, src, static_cast<LabDType>(-maxVal), cols_);
      AscendC::Exp(dst, dst, cols_);
      AscendC::ReduceSum(dstLocal, dst, tmpLocal, cols_);
      LabDType sumVal = dstLocal.GetValue(0);

      /* 第三趟：归一化。整行始终留在片上，三趟之间没有一次 GM 往返 */
      AscendC::Muls(dst, dst, static_cast<LabDType>(1) / sumVal, cols_);

      outQueue.EnQue(dst);
      inQueue.FreeTensor(src);

      AscendC::LocalTensor<LabDType> res = outQueue.DeQue<LabDType>();
      AscendC::DataCopy(yGm[(rowBegin_ + r) * cols_], res, cols_);
      outQueue.FreeTensor(res);
    }
  }

 private:
  AscendC::TPipe pipe;
  AscendC::TQue<AscendC::TPosition::VECIN, QUEUE_DEPTH> inQueue;
  AscendC::TQue<AscendC::TPosition::VECOUT, QUEUE_DEPTH> outQueue;
  AscendC::TBuf<AscendC::TPosition::VECCALC> tmpBuf;
  AscendC::TBuf<AscendC::TPosition::VECCALC> dstBuf;
  AscendC::GlobalTensor<LabDType> xGm, yGm;
  uint32_t cols_ = 0, rowBegin_ = 0, rowCount_ = 0;
};

### 7.6 v5：改用 `SoftMax` 高阶 API

CANN 把 Softmax 做成了高阶 API。用它实现同一件事，核内的三趟收成一行调用：

```cpp
AscendC::SoftMax<LabDType>(dst, src, sharedTmp, tiling, shapeInfo);
```

官方为 `SoftMax` 给出了多个重载，本版用的是**通过 `sharedTmpBuffer` 入参传入临时空间、
不带 `sumTensor` 与 `maxTensor` 参数**的那一个：

> ```
> template <typename T, bool isReuseSource = false, bool isBasicBlock = false,
>           const SoftmaxConfig& config = SOFTMAX_DEFAULT_CFG>
> __aicore__ inline void SoftMax(const LocalTensor<T>& dstTensor,
>                                const LocalTensor<T>& srcTensor,
>                                const LocalTensor<uint8_t>& sharedTmpBuffer,
>                                const SoftMaxTiling& tiling,
>                                const SoftMaxShapeInfo& softmaxShapeInfo = {})
> ```
> ——《Ascend C 算子开发接口》 SoftMax

Atlas A2、A3 训练推理系列产品都在支持列表内，操作数支持 `half` 与 `float` 两种类型。

#### API 内部做的事

官方给出了算法框图。**七个步骤全部在矢量单元上进行**：

<img src="./images/06.07_softmax_api_dataflow.png" alt="06.07_softmax_api_dataflow" width="720px">

*SoftMax 算法框图。绿色为输入输出 Tensor，蓝色为矢量计算*

与第 7.5 节手写的全融合核函数逐步对照：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 步骤 | 高阶 API（官方图 2-80） | 本实验手写版（第 7.5 节） |
| --- | --- | --- |
| ① 求行最大值 | reducemax，[m, k] → [m, 1]，结果留在临时空间 | <code>ReduceMax</code> |
| ② 让行标量作用于整行 | broadcast，[m, 1] → [m, 8]，再对整段做 sub | <code>GetValue</code> 取出标量，再用 <code>Adds</code> |
| ③ 取指数 | exp | <code>Exp</code> |
| ④ 求行和 | reducesum，[m, k] → [m, 1] | <code>ReduceSum</code> |
| ⑤ 归一化 | broadcast 之后对整段做 div | <code>GetValue</code> 取出标量，求倒数后用 <code>Muls</code> |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">步骤</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">高阶 API（官方图 2-80）</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">本实验手写版（第 7.5 节）</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">① 求行最大值</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">reducemax，[m, k] → [m, 1]，结果留在临时空间</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>ReduceMax</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">② 让行标量作用于整行</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">broadcast，[m, 1] → [m, 8]，再对整段做 sub</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>GetValue</code> 取出标量，再用 <code>Adds</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">③ 取指数</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">exp</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>Exp</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">④ 求行和</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">reducesum，[m, k] → [m, 1]</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>ReduceSum</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">⑤ 归一化</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">broadcast 之后对整段做 div</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>GetValue</code> 取出标量，求倒数后用 <code>Muls</code></td>
</tr>
</tbody>
</table>

**唯一的结构性差别在第二步与第五步**：API 用 broadcast 把行标量摊成一个 datablock，
整个过程留在矢量流水里；手写版用 `GetValue` 把它取到标量流水，再作为立即数送回矢量指令。
第 13 节 ⑤ 会用到这一条。

除去输入与输出，还有三个参数需要说明。

**一、`sharedTmp`：交给 API 的临时空间。** 官方把临时空间的处理方式写得很明确：

> 除矩阵计算、HCCL 通信类、卷积计算等，对于多数高阶 API 中临时空间的处理，
> 开发者可以通过 Kernel 侧接口的入参 sharedTmpBuffer 传入提前申请的临时空间、
> 通过接口框架申请临时空间两种方式。
> ——《Ascend C 算子开发指南》 如何使用 Kernel 侧临时空间

`SoftMax` 属于「多数」这一类，因此本实验用 `TBuf` 自行申请、经入参传入。空间给多大也有官方口径：

> 通过 GetSoftMaxMaxTmpSize/GetSoftMaxMinTmpSize 接口获取所需最大和最小临时空间大小，
> **最小空间可以保证功能正确，最大空间用于提升性能**。
> ——《Ascend C 算子开发接口》 SoftMax

两个接口都在主机侧调用。本实验没有调用它们，而是按一行的两倍这个经验值预留——
对本实验的规格够用，但**要交付的算子应当按官方流程定出这两个界**，动手练习第 7 题走一遍。

**二、`shapeInfo`：告诉 API 这块数据的形状。** 官方定义如下：

```cpp
struct SoftMaxShapeInfo {
  uint32_t srcM;     // 非尾轴长度的乘积
  uint32_t srcK;     // 尾轴长度，必须 32Byte 对齐
  uint32_t oriSrcM;  // 原始非尾轴长度的乘积
  uint32_t oriSrcK;  // 原始尾轴长度
};
```

本实验每次交给 API 一行，因此 `srcM = 1`、`srcK = cols`，两个 `ori` 字段与之相同——
**与 v3、v4 逐行处理的口径一致，五个版本因此可以直接相比。**

**三、`tiling`：切分参数。** 官方给出的完整流程是两步：先用 `GetSoftMaxMinTmpSize` /
`GetSoftMaxMaxTmpSize` 定出临时空间的上下界，再用 `SoftMaxTilingFunc` 按 shape 与实际
给出的空间大小算出一份 `SoftMaxTiling`，随 TilingData 送进核函数。官方对第一步另有一句：
「注意该步骤不是必须的，只是作为一个参考，供合理分配计算空间」。

**本版没有走这条路**：`SoftMaxTiling` 保持默认值（结构体各字段初值均为 0），
只把形状经 `shapeInfo` 交给 API。**这一用法有官方依据**，来自模板参数 `config` 的第一个字段：

> ```cpp
> struct SoftmaxConfig {
>   bool isCheckTiling = true;  // 是否需要检查 shape 和 tiling 的一致性；
>                               // 若不一致，API 内会根据 shape 重新计算所需 tiling
>   ...
> };
> constexpr SoftmaxConfig SOFTMAX_DEFAULT_CFG = {true, 0, 0, SoftmaxMode::SOFTMAX_NORMAL};
> ```
> ——《Ascend C 算子开发接口》 SoftMax 模板参数说明

默认配置的 `isCheckTiling` 即为 `true`，因此传入默认值的 tiling 与正确的 `shapeInfo` 时，
API 会检出两者不一致，转而按 shape 重新算一份。**代价是每次调用都要重算**，
主机侧算好传入则只算一次。动手练习第 7 题按官方两步流程重做一遍，把这部分代价量出来。

> **注意 Tiling 结构体在 `AscendC::tiling` 命名空间下，不是 `AscendC`。** 这是官方对所有
> 高阶 API 的统一规定：「所有高阶 API 的 Tiling 结构体定义在 AscendC::tiling 命名空间下」。

#### 约束的核对

官方为 `SoftMax` 列了四条约束，逐条对照本实验的 v5：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 官方约束 | 本实验 v5 的情况 |
| --- | --- |
| src 和 dst 的 Tensor 空间可以复用 | 本版不复用：<code>src</code> 来自输入队列，<code>dst</code> 来自输出队列 |
| <strong>不支持 sharedTmpBuffer 与源操作数和目的操作数地址重叠</strong> | 满足：临时空间由独立的 <code>TBuf</code> 分配 |
| srcTensor 的 last 轴长度需要 32Byte 对齐 | 满足：主规格下一行为 1024 个 <code>float</code>，即 4096 字节 |
| srcM ≠ oriSrcM 或 srcK ≠ oriSrcK 时，须自行把 GM 上的原始输入补齐 | 不涉及：两组值相同，无须补齐 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">官方约束</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">本实验 v5 的情况</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">src 和 dst 的 Tensor 空间可以复用</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">本版不复用：<code>src</code> 来自输入队列，<code>dst</code> 来自输出队列</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>不支持 sharedTmpBuffer 与源操作数和目的操作数地址重叠</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">满足：临时空间由独立的 <code>TBuf</code> 分配</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">srcTensor 的 last 轴长度需要 32Byte 对齐</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">满足：主规格下一行为 1024 个 <code>float</code>，即 4096 字节</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">srcM ≠ oriSrcM 或 srcK ≠ oriSrcK 时，须自行把 GM 上的原始输入补齐</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">不涉及：两组值相同，无须补齐</td>
</tr>
</tbody>
</table>

模板参数 `isBasicBlock` 本版取默认值 `false`。官方给出的启用条件是：尾轴长度 $n < 2048$、
是 64 的倍数，且 `float` 场景下 $n \ge 64$；**同时非尾轴长度的乘积 $m$ 是 8 的倍数**。
本版逐行调用，$m = 1$，不满足后一条。动手练习第 6 题把一次交给 API 的行数改为 8，
两条就同时满足了。


In [ ]:
%%writefile -a src_softmax/ascendc_softmax.asc
/* ===================== v5：用 SoftMax 高阶 API =====================
 * 与 v3、v4 计算的是同一件事，区别在于三趟由高阶 API 内部完成。
 *
 * 高阶 API 需要一份切分参数 SoftMaxTiling。官方给出两条路：
 *   一、主机侧用 SoftMaxTilingFunc 算好，随 TilingData 送进核函数；
 *   二、传入默认值的 SoftMaxTiling 与正确的 SoftMaxShapeInfo——模板参数
 *       config 的 isCheckTiling 默认为 true，API 会检出 tiling 与 shape
 *       不一致，转而按 shape 重新算一份。
 * 本版走第二条，代价是每次调用都要重算一遍。详见 7.6 节。
 * 注意 Tiling 结构体在 AscendC::tiling 命名空间下，不是 AscendC。
 */
class KernelSoftmaxApi {
 public:
  __aicore__ inline KernelSoftmaxApi() {}

  __aicore__ inline void Init(GM_ADDR x, GM_ADDR y, uint32_t cols,
                              uint32_t rowBegin, uint32_t rowCount) {
    cols_ = cols;
    rowBegin_ = rowBegin;
    rowCount_ = rowCount;
    xGm.SetGlobalBuffer(reinterpret_cast<__gm__ LabDType *>(x), 0);
    yGm.SetGlobalBuffer(reinterpret_cast<__gm__ LabDType *>(y), 0);

    pipe.InitBuffer(inQueue, QUEUE_DEPTH, COLS_MAX * sizeof(LabDType));
    pipe.InitBuffer(outQueue, QUEUE_DEPTH, COLS_MAX * sizeof(LabDType));
    /* 交给高阶 API 的临时空间。给得越大，API 能选的切分越宽裕；
     * 这里按一行的两倍预留，足以让它把一行一次算完 */
    pipe.InitBuffer(apiTmpBuf, API_TMP_BYTES);
  }

  __aicore__ inline void Process() {
    AscendC::LocalTensor<uint8_t> sharedTmp = apiTmpBuf.Get<uint8_t>();

    /* 保持默认值的 tiling：结构体各字段初值均为 0，与 shapeInfo 不一致，
     * API 按 isCheckTiling 的默认行为重新算一份 */
    AscendC::tiling::SoftMaxTiling tiling;

    /* 每次交给 API 一行：srcM = 1，srcK = cols，与 v3、v4 的口径一致 */
    AscendC::SoftMaxShapeInfo shapeInfo;
    shapeInfo.srcM = 1;
    shapeInfo.srcK = cols_;
    shapeInfo.oriSrcM = 1;
    shapeInfo.oriSrcK = cols_;

    for (uint32_t r = 0; r < rowCount_; ++r) {
      AscendC::LocalTensor<LabDType> xLocal = inQueue.AllocTensor<LabDType>();
      AscendC::DataCopy(xLocal, xGm[(rowBegin_ + r) * cols_], cols_);
      inQueue.EnQue(xLocal);

      AscendC::LocalTensor<LabDType> src = inQueue.DeQue<LabDType>();
      AscendC::LocalTensor<LabDType> dst = outQueue.AllocTensor<LabDType>();

      /* 三趟收在这一行调用之内：求最大值、减最大值取指数并求和、归一化 */
      AscendC::SoftMax<LabDType>(dst, src, sharedTmp, tiling, shapeInfo);

      outQueue.EnQue(dst);
      inQueue.FreeTensor(src);

      AscendC::LocalTensor<LabDType> res = outQueue.DeQue<LabDType>();
      AscendC::DataCopy(yGm[(rowBegin_ + r) * cols_], res, cols_);
      outQueue.FreeTensor(res);
    }
  }

 private:
  AscendC::TPipe pipe;
  AscendC::TQue<AscendC::TPosition::VECIN, QUEUE_DEPTH> inQueue;
  AscendC::TQue<AscendC::TPosition::VECOUT, QUEUE_DEPTH> outQueue;
  AscendC::TBuf<AscendC::TPosition::VECCALC> apiTmpBuf;
  AscendC::GlobalTensor<LabDType> xGm, yGm;
  uint32_t cols_ = 0, rowBegin_ = 0, rowCount_ = 0;
};

### 7.7 行切分与核函数入口

切分单位是**行**而不是元素。第 $i$ 行的起始地址是 $i \times N \times 4$ 字节，只要 $N \times 4$ 是 32 的倍数，任意行的起始地址必然对齐，**因此本实验无须像实验二那样做向下对齐处理**。

这个前提等价于要求 **$N$ 是 8 的倍数**（主规格 $N = 1024$ 满足）。形状由命令行给出，因此主程序在开头连同行数上界一起检查，不满足即报错退出——**前提要在代码里落实，不能只写在正文里**。

行数不能被核数整除时，前 `rem` 个核各多分一行，各核的行数相差不超过一。

<img src="./images/06.07_split_rows.png" alt="06.07_split_rows"  width="960px" >

六个入口都走同一条路：先按 `GetBlockIdx()` 算出本核负责的行区间，再交给对应的算子类。**行数少于核数时，靠后的核分不到行**，此时直接返回，不再进入算子类——否则第一趟那段补位写出会以零长度下发。`KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY)` 声明这些核函数只用矢量单元。

In [ ]:
%%writefile -a src_softmax/ascendc_softmax.asc
/* ===================== 行切分：把 rows 行分给 blockNum 个核 =====================
 * 切分单位是行而非元素。只要 cols x 4 是 32 的倍数，
 * 任意行的起始地址必然对齐，因此无须像实验二 v4 那样做向下对齐处理。
 * 行数不能被核数整除时，前 rem 个核各多分一行。
 */
__aicore__ inline void SplitRows(uint32_t rows, uint32_t blockNum,
                                 uint32_t blockIdx, uint32_t *rowBegin,
                                 uint32_t *rowCount) {
  uint32_t base = rows / blockNum;
  uint32_t rem = rows % blockNum;
  if (blockIdx < rem) {
    *rowCount = base + 1;
    *rowBegin = blockIdx * (base + 1);
  } else {
    *rowCount = base;
    *rowBegin = rem * (base + 1) + (blockIdx - rem) * base;
  }
}

/* ===================== 六个核函数入口 =====================
 * 六个入口都走同一条路：先按 GetBlockIdx() 算出本核负责的行区间，
 * 再交给对应的算子类。行数少于核数时，靠后的核分不到行，直接返回。
 */

extern "C" __global__ __aicore__ void softmax_rowmax(GM_ADDR x, GM_ADDR m,
                                                     uint32_t rows,
                                                     uint32_t cols) {
  KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY);
  uint32_t rowBegin = 0, rowCount = 0;
  SplitRows(rows, AscendC::GetBlockNum(), AscendC::GetBlockIdx(), &rowBegin,
            &rowCount);
  if (rowCount == 0) {
    return; /* 行数少于核数时，靠后的核分不到行，直接返回 */
  }
  KernelRowMax op;
  op.Init(x, m, cols, rowBegin, rowCount);
  op.Process();
}

extern "C" __global__ __aicore__ void softmax_expsum_nomax(GM_ADDR x, GM_ADDR m,
                                                           GM_ADDR t, GM_ADDR s,
                                                           uint32_t rows,
                                                           uint32_t cols) {
  KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY);
  uint32_t rowBegin = 0, rowCount = 0;
  SplitRows(rows, AscendC::GetBlockNum(), AscendC::GetBlockIdx(), &rowBegin,
            &rowCount);
  if (rowCount == 0) {
    return; /* 行数少于核数时，靠后的核分不到行，直接返回 */
  }
  KernelExpSum<false> op;
  op.Init(x, m, t, s, cols, rowBegin, rowCount);
  op.Process();
}

extern "C" __global__ __aicore__ void softmax_expsum_submax(
    GM_ADDR x, GM_ADDR m, GM_ADDR t, GM_ADDR s, uint32_t rows, uint32_t cols) {
  KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY);
  uint32_t rowBegin = 0, rowCount = 0;
  SplitRows(rows, AscendC::GetBlockNum(), AscendC::GetBlockIdx(), &rowBegin,
            &rowCount);
  if (rowCount == 0) {
    return; /* 行数少于核数时，靠后的核分不到行，直接返回 */
  }
  KernelExpSum<true> op;
  op.Init(x, m, t, s, cols, rowBegin, rowCount);
  op.Process();
}

extern "C" __global__ __aicore__ void softmax_divide(GM_ADDR t, GM_ADDR s,
                                                     GM_ADDR y, uint32_t rows,
                                                     uint32_t cols) {
  KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY);
  uint32_t rowBegin = 0, rowCount = 0;
  SplitRows(rows, AscendC::GetBlockNum(), AscendC::GetBlockIdx(), &rowBegin,
            &rowCount);
  if (rowCount == 0) {
    return; /* 行数少于核数时，靠后的核分不到行，直接返回 */
  }
  KernelDivide op;
  op.Init(t, s, y, cols, rowBegin, rowCount);
  op.Process();
}

extern "C" __global__ __aicore__ void softmax_fused(GM_ADDR x, GM_ADDR y,
                                                    uint32_t rows,
                                                    uint32_t cols) {
  KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY);
  uint32_t rowBegin = 0, rowCount = 0;
  SplitRows(rows, AscendC::GetBlockNum(), AscendC::GetBlockIdx(), &rowBegin,
            &rowCount);
  if (rowCount == 0) {
    return; /* 行数少于核数时，靠后的核分不到行，直接返回 */
  }
  KernelSoftmaxFused op;
  op.Init(x, y, cols, rowBegin, rowCount);
  op.Process();
}

extern "C" __global__ __aicore__ void softmax_api(GM_ADDR x, GM_ADDR y,
                                                  uint32_t rows,
                                                  uint32_t cols) {
  KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY);
  uint32_t rowBegin = 0, rowCount = 0;
  SplitRows(rows, AscendC::GetBlockNum(), AscendC::GetBlockIdx(), &rowBegin,
            &rowCount);
  if (rowCount == 0) {
    return; /* 行数少于核数时，靠后的核分不到行，直接返回 */
  }
  KernelSoftmaxApi op;
  op.Init(x, y, cols, rowBegin, rowCount);
  op.Process();
}

## 8. Host 侧实现

### 8.1 工具函数

六件事：返回值检查、计时、造数、参考真值、CPU 基准、结果校验。

**参考真值全程用 `double`。** 三项判据里的绝对误差以它为准，因此它必须比被测对象精确若干个数量级。它同样先减去行最大值——`double` 的上界虽然远大于 `float32`，但这里要的是一个与实现无关的正确结果。

**CPU 基准用 `float` 且算法与 v3 相同**：全融合、逐行、单线程。加速比以它为分母。

`Verify` 一次返回三项：最大绝对误差、各行求和与 1 的最大偏差、以及输出是否全为有限值。三项的排列顺序与第 3 节一致。

**非有限的元素被整个跳过**：既不进入误差统计，也不进入行和统计。跳过误差统计是因为 `nan` 会污染最大值的比较；跳过行和统计则使被跳过的元素个数反映在行和的偏差里——**一行里只要有元素被跳过，它的和就不再是 1**。

In [ ]:
%%writefile -a src_softmax/ascendc_softmax.asc
/* ===================== Host 侧：工具函数 ===================== */

#define ACL_CHECK(expr)                                                        \
  do {                                                                         \
    aclError _r = (expr);                                                      \
    if (_r != ACL_SUCCESS) {                                                   \
      std::printf("[ACL ERROR] %s:%d %s 返回 %d\n", __FILE__, __LINE__, #expr, \
                  static_cast<int32_t>(_r));                                   \
      std::exit(1);                                                            \
    }                                                                          \
  } while (0)

static aclrtStream g_stream = nullptr;

static double NowMs() {
  timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return ts.tv_sec * 1e3 + ts.tv_nsec * 1e-6;
}

/* 线性同余发生器：与实验二至实验五相同，保证同一 seed 在任何机器上数据一致 */
static void GenData(std::vector<float> &v, uint32_t seed, float amp) {
  uint32_t st = seed;
  for (size_t i = 0; i < v.size(); ++i) {
    st = st * 1103515245u + 12345u;
    float u = static_cast<float>((st >> 8) & 0xFFFFFFu) / 16777216.0f;
    v[i] = (u * 2.0f - 1.0f) * amp;
  }
}

/* 逐行 Softmax 的参考实现，全程 double：本实验的真值 */
static void SoftmaxRef(const std::vector<float> &x, std::vector<double> &y,
                       uint32_t rows, uint32_t cols) {
  for (uint32_t i = 0; i < rows; ++i) {
    const float *xr = x.data() + static_cast<size_t>(i) * cols;
    double *yr = y.data() + static_cast<size_t>(i) * cols;
    double m = xr[0];
    for (uint32_t j = 1; j < cols; ++j) {
      if (static_cast<double>(xr[j]) > m) m = xr[j];
    }
    double s = 0.0;
    for (uint32_t j = 0; j < cols; ++j) {
      yr[j] = std::exp(static_cast<double>(xr[j]) - m);
      s += yr[j];
    }
    for (uint32_t j = 0; j < cols; ++j) yr[j] /= s;
  }
}

/* CPU 基准：float 全融合实现，与 v3 的算法完全一致，用作加速比的分母 */
static void SoftmaxCpu(const std::vector<float> &x, std::vector<float> &y,
                       uint32_t rows, uint32_t cols) {
  for (uint32_t i = 0; i < rows; ++i) {
    const float *xr = x.data() + static_cast<size_t>(i) * cols;
    float *yr = y.data() + static_cast<size_t>(i) * cols;
    float m = xr[0];
    for (uint32_t j = 1; j < cols; ++j) {
      if (xr[j] > m) m = xr[j];
    }
    float s = 0.0f;
    for (uint32_t j = 0; j < cols; ++j) {
      yr[j] = std::exp(xr[j] - m);
      s += yr[j];
    }
    float inv = 1.0f / s;
    for (uint32_t j = 0; j < cols; ++j) yr[j] *= inv;
  }
}

/* 三项判据：与真值的最大绝对误差、输出是否全为有限值、逐行求和与 1 的偏差。
 * 非有限的元素被跳过，因此它既不进入误差统计，也不进入行和统计 */
struct Check {
  double maxAbsErr; /* 与 double 真值的最大绝对误差 */
  double rowSumErr; /* 各行求和与 1 的最大偏差 */
  int32_t finite;   /* 输出中是否不含 inf 与 nan */
};

static Check Verify(const std::vector<float> &y, const std::vector<double> &ref,
                    uint32_t rows, uint32_t cols) {
  Check c = {0.0, 0.0, 1};
  for (uint32_t i = 0; i < rows; ++i) {
    const float *yr = y.data() + static_cast<size_t>(i) * cols;
    const double *rr = ref.data() + static_cast<size_t>(i) * cols;
    double rowSum = 0.0;
    for (uint32_t j = 0; j < cols; ++j) {
      if (!std::isfinite(yr[j])) {
        c.finite = 0;
        continue;
      }
      rowSum += yr[j];
      double e = std::fabs(static_cast<double>(yr[j]) - rr[j]);
      if (e > c.maxAbsErr) c.maxAbsErr = e;
    }
    double d = std::fabs(rowSum - 1.0);
    if (d > c.rowSumErr) c.rowSumErr = d;
  }
  return c;
}

/* 重复计时：预热若干次，再重复若干次取平均 */
template <typename F>
static double TimeIt(F fn, int32_t warmup, int32_t repeat) {
  for (int32_t i = 0; i < warmup; ++i) fn();
  ACL_CHECK(aclrtSynchronizeStream(g_stream));
  double t0 = NowMs();
  for (int32_t i = 0; i < repeat; ++i) fn();
  ACL_CHECK(aclrtSynchronizeStream(g_stream));
  return (NowMs() - t0) / repeat;
}

### 8.2 主程序

顺序与前几个实验一致：造数 → 算真值与基准 → 搬到设备 → 逐版本运行、计时、校验 → 打印。

三处与前几个实验不同：

- **形状与输入幅度由命令行给出**，因此第 11 节的量程扫描与第 12 节的行数泛化都不必重新编译；只有改动核数或列数上界这类编译期常量时才需要重编。
- **每个版本运行前把输出缓冲清零**，这样没有被写到与写错了在结果上可以区分：若某个核的区间根本没有被写入，留下的 0 会同时触发绝对误差与行和两项判据。
- **五个版本的下发过程各写成一个 lambda**，计时函数只负责重复调用，两者互不干扰。

判定用的两个容差都打印在 `[BASE]` 行：`atol` 由第 3.1 节那条公式算出，`rowsum_tol` 按第 3.3 节的量级取 $10^{-4}$。**三项判据全部通过才算通过**，但三项各自的数值都单独打印，因此第 11 节可以分别追踪它们。

In [ ]:
%%writefile -a src_softmax/ascendc_softmax.asc
/* ===================== Host 侧：主程序 ===================== */

int32_t main(int32_t argc, char **argv) {
  const uint32_t rows =
      (argc > 1) ? static_cast<uint32_t>(std::atoi(argv[1])) : ROWS_DEFAULT;
  const uint32_t cols =
      (argc > 2) ? static_cast<uint32_t>(std::atoi(argv[2])) : COLS_MAX;
  const float amp = (argc > 3) ? static_cast<float>(std::atof(argv[3]))
                               : static_cast<float>(LAB_AMP);

  if (cols > COLS_MAX) {
    std::printf("[ERROR] cols=%u 超出编译期上界 COLS_MAX=%u\n", cols, COLS_MAX);
    return 1;
  }
  if (rows > ROWS_MAX) {
    std::printf("[ERROR] rows=%u 超出编译期上界 ROWS_MAX=%u\n", rows, ROWS_MAX);
    return 1;
  }
  /* 行首地址天然对齐的前提：每行字节数是 32 的倍数（第 7.7 节） */
  if (cols % ALIGN_ELEM != 0) {
    std::printf("[ERROR] cols=%u 不是 %u 的倍数，行首地址无法对齐\n", cols,
                ALIGN_ELEM);
    return 1;
  }

  const size_t nElem = static_cast<size_t>(rows) * cols;
  const size_t nBytes = nElem * sizeof(LabDType);
  /* 行标量向量按 32 字节对齐向上取整，避免核间写出时越界 */
  const size_t rowsPad =
      ((rows + ALIGN_ELEM - 1) / ALIGN_ELEM) * ALIGN_ELEM + ALIGN_ELEM;
  const size_t rowBytes = rowsPad * sizeof(LabDType);

  /* ---- 造数与真值 ---- */
  std::vector<float> hostX(nElem);
  GenData(hostX, 12345u, amp);

  float maxAbsX = 0.0f;
  for (size_t i = 0; i < nElem; ++i) {
    float a = std::fabs(hostX[i]);
    if (a > maxAbsX) maxAbsX = a;
  }

  std::vector<double> ref(nElem);
  SoftmaxRef(hostX, ref, rows, cols);

  std::vector<float> hostCpu(nElem);
  double t0 = NowMs();
  SoftmaxCpu(hostX, hostCpu, rows, cols);
  double cpuMs = NowMs() - t0;
  Check cpuChk = Verify(hostCpu, ref, rows, cols);

  /* 判定容差：与实验二至实验五同一条公式。Softmax 的输出上界为 1，
   * 累加链长度为 cols，因此取 atol = c * eps * sqrt(cols) * 1 */
  const double EPS = 1.1920929e-7;
  const double atol = 2.0 * EPS * std::sqrt(static_cast<double>(cols));
  /* 第三项判据：逐行求和为 1。这是 Softmax 的定义所蕴含的不变量，
   * 与逐元素比对相互独立。行和是 cols 个 float32 输出在 double 中累加，
   * 其误差量级约为 eps * sqrt(cols)，本实验取 1e-4，比它宽两个数量级 */
  const double ROWSUM_TOL = 1e-4;

  std::printf(
      "[BASE] rows=%u cols=%u amp=%.1f max_abs_x=%.3f cpu_ms=%.4f "
      "cpu_max_abs_err=%.3e cpu_rowsum_err=%.3e atol=%.3e rowsum_tol=%.3e\n",
      rows, cols, amp, maxAbsX, cpuMs, cpuChk.maxAbsErr, cpuChk.rowSumErr, atol,
      ROWSUM_TOL);

  /* ---- 设备内存 ---- */
  ACL_CHECK(aclInit(nullptr));
  ACL_CHECK(aclrtSetDevice(0));
  ACL_CHECK(aclrtCreateStream(&g_stream));

  /* 设备指针声明为 uint8_t*，可直接作为 GM_ADDR 实参下发，无须显式转换 */
  uint8_t *devX = nullptr, *devY = nullptr, *devT = nullptr;
  uint8_t *devM = nullptr, *devS = nullptr;
  ACL_CHECK(aclrtMalloc((void **)&devX, nBytes, ACL_MEM_MALLOC_HUGE_FIRST));
  ACL_CHECK(aclrtMalloc((void **)&devY, nBytes, ACL_MEM_MALLOC_HUGE_FIRST));
  ACL_CHECK(aclrtMalloc((void **)&devT, nBytes, ACL_MEM_MALLOC_HUGE_FIRST));
  ACL_CHECK(aclrtMalloc((void **)&devM, rowBytes, ACL_MEM_MALLOC_HUGE_FIRST));
  ACL_CHECK(aclrtMalloc((void **)&devS, rowBytes, ACL_MEM_MALLOC_HUGE_FIRST));
  ACL_CHECK(aclrtMemcpy(devX, nBytes, hostX.data(), nBytes,
                        ACL_MEMCPY_HOST_TO_DEVICE));

  std::vector<float> hostY(nElem);

  /* 五个版本的下发过程。每个 lambda 只负责下发，不负责同步 */
  auto launchV1 = [&]() {
    softmax_expsum_nomax<<<1, nullptr, g_stream>>>(devX, devM, devT, devS, rows,
                                                   cols);
    softmax_divide<<<1, nullptr, g_stream>>>(devT, devS, devY, rows, cols);
  };
  auto launchV2 = [&]() {
    softmax_rowmax<<<1, nullptr, g_stream>>>(devX, devM, rows, cols);
    softmax_expsum_submax<<<1, nullptr, g_stream>>>(devX, devM, devT, devS,
                                                    rows, cols);
    softmax_divide<<<1, nullptr, g_stream>>>(devT, devS, devY, rows, cols);
  };
  auto launchV3 = [&]() {
    softmax_fused<<<1, nullptr, g_stream>>>(devX, devY, rows, cols);
  };
  auto launchV4 = [&]() {
    softmax_fused<<<BLOCK_DIM, nullptr, g_stream>>>(devX, devY, rows, cols);
  };
  auto launchV5 = [&]() {
    softmax_api<<<BLOCK_DIM, nullptr, g_stream>>>(devX, devY, rows, cols);
  };

  /* 访存量（MiB）：非融合读三次写两次，融合读一次写一次 */
  const double MiB = 1024.0 * 1024.0;
  double bytesFused = 2.0 * nBytes / MiB;
  double bytesV1 = 4.0 * nBytes / MiB; /* 第二趟读写各一次，第三趟读写各一次 */
  double bytesV2 = 5.0 * nBytes / MiB; /* 第一趟多读一次 x */

  constexpr int32_t NVER = 5;
  double kernelMs[NVER] = {0, 0, 0, 0, 0};
  Check chk[NVER];
  const char *names[NVER] = {"v1", "v2", "v3", "v4", "v5"};
  const double bytesArr[NVER] = {bytesV1, bytesV2, bytesFused, bytesFused,
                                 bytesFused};
  const uint32_t blocksArr[NVER] = {1, 1, 1, BLOCK_DIM, BLOCK_DIM};

  for (int32_t v = 0; v < NVER; ++v) {
    ACL_CHECK(aclrtMemset(devY, nBytes, 0, nBytes));

    auto run = [&]() {
      if (v == 0)
        launchV1();
      else if (v == 1)
        launchV2();
      else if (v == 2)
        launchV3();
      else if (v == 3)
        launchV4();
      else
        launchV5();
    };

    /* 本实验只取核函数口径：数据已在设备上，只计下发与执行 */
    kernelMs[v] = TimeIt(run, WARMUP, REPEAT);

    ACL_CHECK(aclrtMemcpy(hostY.data(), nBytes, devY, nBytes,
                          ACL_MEMCPY_DEVICE_TO_HOST));
    chk[v] = Verify(hostY, ref, rows, cols);

    /* 三项判据全部通过才算通过。哪一项先失效，本身就是有信息量的 */
    bool okFinite = (chk[v].finite == 1);
    bool okAtol = (chk[v].maxAbsErr <= atol);
    bool okRowSum = (chk[v].rowSumErr <= ROWSUM_TOL);
    bool pass = okFinite && okAtol && okRowSum;
    std::printf(
        "[VERIFY] ver=%s atol=%.3e max_abs_err=%.3e rowsum_err=%.3e "
        "finite=%d result=%s\n",
        names[v], atol, chk[v].maxAbsErr, chk[v].rowSumErr, chk[v].finite,
        pass ? "PASS" : "FAIL");
  }

  for (int32_t v = 0; v < NVER; ++v) {
    std::printf("[PERF] ver=%s blocks=%u bytes_mib=%.2f kernel_ms=%.4f "
                "sp_kernel=%.2f\n",
                names[v], blocksArr[v], bytesArr[v], kernelMs[v],
                cpuMs / kernelMs[v]);
  }

  ACL_CHECK(aclrtFree(devX));
  ACL_CHECK(aclrtFree(devY));
  ACL_CHECK(aclrtFree(devT));
  ACL_CHECK(aclrtFree(devM));
  ACL_CHECK(aclrtFree(devS));
  ACL_CHECK(aclrtDestroyStream(g_stream));
  ACL_CHECK(aclrtResetDevice(0));
  ACL_CHECK(aclFinalize());

  std::printf("[Success]\n");
  return 0;
}

## 9. 编译与运行

一条 `bisheng` 命令。编译选项集中定义一处，便于按需改动。

In [ ]:
import subprocess

ARCH = "dav-2201"  # ← 若设备不是 Atlas A2/A3，请按实验一 §7.2 的表修改
SRC = "src_softmax/ascendc_softmax.asc"
EXE = "src_softmax/ascendc_softmax"

FLAGS = ["--npu-arch=" + ARCH, "-O2", "-lm"]

# bisheng [算子源文件] [编译选项] -o [输出产物名称]
cmd = ["bisheng", SRC] + FLAGS + ["-o", EXE]
print("$ " + " ".join(cmd))

proc = subprocess.run(cmd, capture_output=True, text=True)
msg = (proc.stdout + proc.stderr).strip()
if msg:
    print(msg)

print(
    "✅ 编译成功"
    if proc.returncode == 0
    else "❌ 编译失败（返回码 %d）" % proc.returncode
)

程序在下发之前先在主机侧算两次参考：一次 `double` 真值、一次 `float` 基准，两次都是单线程的全量计算。

In [ ]:
def run_demo(args=(), exe=None, timeout=1800):
    # 与实验二至实验六同名同约定：只返回 stdout
    proc = subprocess.run(
        ["./" + (exe or EXE)] + [str(a) for a in args],
        capture_output=True,
        text=True,
        timeout=timeout,
    )
    if proc.returncode != 0 and not proc.stdout:
        print("返回码", proc.returncode)
        print(proc.stderr)
    return proc.stdout


out_main = run_demo()
print(out_main)

### 9.1 解析输出

In [ ]:
def parse_rows(text, tag):
    # 把所有以 [tag] 开头的记录行解析为 dict 列表
    rows = []
    for line in text.splitlines():
        if line.startswith("[" + tag + "]"):
            d = {}
            for kv in line.split()[1:]:
                k, v = kv.split("=", 1)
                d[k] = v if k in ("ver", "result") else float(v)
            rows.append(d)
    return rows


def parse_perf(text):
    return parse_rows(text, "PERF")


def parse_verify(text):
    return {r["ver"]: r for r in parse_rows(text, "VERIFY")}


def parse_base(text):
    rows = parse_rows(text, "BASE")
    return rows[0] if rows else None


if not parse_perf(out_main):
    raise RuntimeError("输出里没有 [PERF] 记录行，请回看上一个单元格的输出")

base_main = parse_base(out_main)
rows_main = parse_perf(out_main)
chk_main = parse_verify(out_main)

print(
    "规格：%d × %d    输入幅度 ±%.0f    实际 max|x| = %.3f"
    % (base_main["rows"], base_main["cols"], base_main["amp"], base_main["max_abs_x"])
)
print(
    "CPU 基准（float，与 v3 同算法）：%.2f ms    自身最大绝对误差 %.3e"
    % (base_main["cpu_ms"], base_main["cpu_max_abs_err"])
)
print(
    "判据：atol = %.3e    行和容差 = %.3e"
    % (base_main["atol"], base_main["rowsum_tol"])
)
print()

hdr = (
    "版本",
    "核数",
    "访存/MiB",
    "kernel(ms)",
    "vs CPU",
    "最大绝对误差",
    "行和偏差",
    "有限",
    "判定",
)
print("%-5s %5s %10s %12s %9s %14s %12s %6s %7s" % hdr)
print("-" * 92)
for r in rows_main:
    v = chk_main[r["ver"]]
    print(
        "%-5s %5d %10.1f %12.4f %8.1fx %14.3e %12.3e %6d %7s"
        % (
            r["ver"],
            r["blocks"],
            r["bytes_mib"],
            r["kernel_ms"],
            r["sp_kernel"],
            v["max_abs_err"],
            v["rowsum_err"],
            v["finite"],
            v["result"],
        )
    )
print()
print("注：v1 不做数值稳定化处理，在主规格的输入幅度下判定为 FAIL 属预期结果，"
      "原因见第 6 节，量程边界见第 11 节。")

可直接由上表读出的四个量：

- **v2 与 v1 的耗时之差**，即数值稳定化的代价；
- **v2 与 v3 的比值**，与第 1.3 节的 $2.50$ 对照；
- **v4 与 v3 的比值**，即沿行切分的扩展性；
- **v5 与 v4 的比值**，即高阶 API 与手写实现的差距。

In [ ]:
by = {r["ver"]: r for r in rows_main}

d_stab = by["v2"]["kernel_ms"] - by["v1"]["kernel_ms"]
print(
    "数值稳定化的代价：v2 − v1 = %.4f ms（占 v2 的 %.0f%%）"
    % (d_stab, 100.0 * d_stab / by["v2"]["kernel_ms"])
)
print(
    "融合的收益      ：v2 / v3 = %.2fx    访存量之比 = %.2f"
    % (
        by["v2"]["kernel_ms"] / by["v3"]["kernel_ms"],
        by["v2"]["bytes_mib"] / by["v3"]["bytes_mib"],
    )
)
print(
    "多核的收益      ：v3 / v4 = %.2fx    核数之比 = %d"
    % (by["v3"]["kernel_ms"] / by["v4"]["kernel_ms"], int(by["v4"]["blocks"]))
)
print(
    "手写 vs 高阶 API：v4 / v5 = %.2fx    两者核数与访存量相同"
    % (by["v4"]["kernel_ms"] / by["v5"]["kernel_ms"])
)

## 10. 结果可视化

左图是五个版本的核函数耗时，右图把实测加速比与访存量之比并排，两者都以 **v2 为基准**——v2 与 v3 的算法完全相同，只有组织方式不同，因此这一对的比值才直接对应第 1.3 节的 $2.50$。

**v4 与 v5 的柱子不构成访存对照**：它们与 v3 的差别是核数与实现方式，不是访存量，那两根柱子反映的是并行度与实现效率。

In [ ]:
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

matplotlib.rcParams["font.sans-serif"] = ["DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
C_KERNEL, C_E2E, C_BASE, C_OPT = "#3B6FE0", "#E07A3B", "#9AA5B1", "#2E9E6B"

vers = [r["ver"] for r in rows_main]
kms = [r["kernel_ms"] for r in rows_main]
xs = np.arange(len(vers))

fig, ax = plt.subplots(1, 2, figsize=(12, 4.4))

ax[0].bar(xs, kms, 0.5, label="kernel", color=C_KERNEL)
ax[0].axhline(base_main["cpu_ms"], color=C_BASE, ls="--", lw=1.4, label="CPU")
ax[0].set_xticks(xs)
ax[0].set_xticklabels(vers)
ax[0].set_ylabel("ms")
ax[0].set_yscale("log")
ax[0].set_title("kernel time (log scale)")
ax[0].legend()
ax[0].grid(axis="y", alpha=0.3, ls=":")

base_ms = by["v2"]["kernel_ms"]  # 以 v2 为基准：它与 v3 的算法相同，只是组织方式不同
meas = [base_ms / r["kernel_ms"] for r in rows_main]
theo = [by["v2"]["bytes_mib"] / r["bytes_mib"] for r in rows_main]
ax[1].bar(xs - 0.2, meas, 0.4, label="measured", color=C_OPT)
ax[1].bar(xs + 0.2, theo, 0.4, label="bytes ratio", color=C_BASE)
ax[1].set_xticks(xs)
ax[1].set_xticklabels(vers)
ax[1].set_ylabel("x  (relative to v2)")
ax[1].set_title("speedup vs memory-traffic ratio")
ax[1].legend()
ax[1].grid(axis="y", alpha=0.3, ls=":")

plt.tight_layout()
plt.show()

## 11. 量程扫描：`exp` 的溢出边界

第 1.2 节由 `float32` 的表示上界推出：$e^{x}$ 在 $x > 88.7$ 时上溢。下面沿输入幅度扫描，**用同一个可执行文件、不重新编译**，看 v1 从哪一档开始失效，以及**三项判据分别在哪一档失效**。

v2 全程作为对照：它减去了行最大值，指数的自变量恒不大于 0，因而不受输入幅度影响。

In [ ]:
AMPS = [20, 40, 60, 80, 85, 87, 90, 95, 100, 120]
SCAN_ROWS, SCAN_COLS = 256, 1024

scan = []
for a in AMPS:
    out = run_demo([SCAN_ROWS, SCAN_COLS, a])
    b = parse_base(out)
    c = parse_verify(out)
    if b is None or "v1" not in c:
        print("幅度 %d 未取到结果，已跳过" % a)
        continue
    scan.append((a, b, c["v1"], c["v2"]))
    print(
        "幅度 ±%-4d  max|x| = %7.3f   v1 %s   v2 %s"
        % (a, b["max_abs_x"], c["v1"]["result"], c["v2"]["result"])
    )

把三项判据分开列出。**表中要看的是它们失效的先后**，而不是具体数值。

In [ ]:
print(
    "%-8s %10s %14s %12s %8s   %s"
    % ("幅度", "max|x|", "最大绝对误差", "行和偏差", "有限", "失效的判据")
)
print("-" * 88)
for a, b, v1, v2 in scan:
    fired = []
    if v1["finite"] == 0:
        fired.append("有限性")
    if v1["max_abs_err"] > b["atol"]:
        fired.append("绝对误差")
    if v1["rowsum_err"] > b["rowsum_tol"]:
        fired.append("行和")
    print(
        "%-8s %10.3f %14.3e %12.3e %8d   %s"
        % (
            "±%d" % a,
            b["max_abs_x"],
            v1["max_abs_err"],
            v1["rowsum_err"],
            v1["finite"],
            "、".join(fired) if fired else "— 全部通过",
        )
    )

左图是两个版本各自的判定结果随 `max|x|` 的变化，红色虚线是 $\ln(\text{FLT\_MAX}) = 88.7$ 这条理论边界；右图把三项判据分开画，**看的是三条线抬起的先后**。

In [ ]:
amps = [a for a, _, _, _ in scan]
mx = [b["max_abs_x"] for _, b, _, _ in scan]

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))

ok1 = [1 if v1["result"] == "PASS" else 0 for _, _, v1, _ in scan]
ok2 = [1 if v2["result"] == "PASS" else 0 for _, _, _, v2 in scan]
ax[0].step(mx, ok1, where="post", color=C_KERNEL, lw=2.0, label="v1 (no max sub)")
ax[0].step(mx, ok2, where="post", color=C_OPT, lw=2.0, ls="--", label="v2 (max sub)")
ax[0].axvline(88.72, color="#C7000B", ls=":", lw=1.6)
ax[0].text(88.72, 0.5, "  ln(FLT_MAX) = 88.7", color="#C7000B", fontsize=9)
ax[0].set_ylim(-0.15, 1.2)
ax[0].set_yticks([0, 1])
ax[0].set_yticklabels(["FAIL", "PASS"])
ax[0].set_xlabel("max |x|")
ax[0].set_title("where the no-max-sub version breaks")
ax[0].legend(loc="lower left")
ax[0].grid(alpha=0.3, ls=":")

for key, col, lab in (
    ("finite", "#C7000B", "non-finite"),
    ("max_abs_err", C_KERNEL, "abs err"),
    ("rowsum_err", C_E2E, "row-sum err"),
):
    if key == "finite":
        ys = [0 if v1["finite"] == 1 else 1 for _, _, v1, _ in scan]
        ax[1].step(mx, ys, where="post", color=col, lw=2.0, label=lab)
    else:
        tol = scan[0][1]["atol"] if key == "max_abs_err" else scan[0][1]["rowsum_tol"]
        ys = [1 if v1[key] > tol else 0 for _, _, v1, _ in scan]
        ax[1].step(mx, ys, where="post", color=col, lw=2.0, label=lab)
ax[1].set_ylim(-0.15, 1.6)
ax[1].set_yticks([0, 1])
ax[1].set_yticklabels(["quiet", "fires"])
ax[1].set_xlabel("max |x|")
ax[1].set_title("which check fires, and where")
ax[1].legend(loc="upper left")
ax[1].grid(alpha=0.3, ls=":")

plt.tight_layout()
plt.show()

## 12. 行数泛化

主规格的 2048 行能被常用的核数整除，这掩盖了尾块的处理。改用几个不能整除的行数，
确认第 7.7 节的切分逻辑正确。形状由命令行给出，因此这一节不必重新编译。

In [ ]:
for rows in (2050, 2047, 17):
    out = run_demo([rows, 1024])
    v = parse_verify(out)
    b = parse_base(out)
    if not v:
        print("%5d 行：未取到结果" % rows)
        continue
    flags = " ".join("%s=%s" % (k, v[k]["result"]) for k in ("v2", "v3", "v4", "v5"))
    print("%5d 行（%d 核）   %s" % (rows, int(parse_perf(out)[3]["blocks"]), flags))

## 13. 结果分析

> 本节给出的是判读方法与定性规律。具体数值与设备型号、CANN 版本以及运行时的负载有关，不同机器上测得的数并不相同；应以实际运行得到的输出为准。

**① 数值稳定化的代价，以及它为什么必须付**

v1 与 v2 的算法差别只有一趟 `ReduceMax` 与一条 `Adds`，因此两者的耗时之差就是这一处理的代价。

按访存量估计，v2 比 v1 多读一遍 $x$，多出的访存约占四分之一。**实测的相对增幅通常明显大于这个比例**：多出的那一趟是一个独立的核函数，要多一次下发；它还要对整个张量做一遍 `ReduceMax`，而这是访存模型算不到的计算。**当实测增幅显著超过访存量的增幅时，说明代价不只落在搬运上**——这本身就是一条判读方法。

第 11 节表明，不付这个代价，输入幅度一旦越过 `float32` 的量程，结果就完全不可用。**在算子里，正确性不是与性能并列的一项指标，而是性能得以被讨论的前提。**

**② 融合的收益为什么达不到访存量之比**

第 10 节右图把实测加速比与访存量之比并排。两者接近，说明这段计算是访存受限的，融合省下的访存直接变成了时间；实测明显低于访存量之比，说明还有别的瓶颈。

**最干净的一对是 v2 与 v3**：两者做的矢量计算完全相同，只有组织方式不同，因此比值可以直接与第 1.3 节的上界对照。

v1 与 v3 提供了一条旁证：**v3 的访存量只有 v1 的一半，矢量计算却比 v1 多两趟**。若两者的单核耗时相差无几，说明这两项的影响量级相当——**减少的搬运与增加的计算彼此抵消**。这也解释了 v2 到 v3 为什么达不到第 1.3 节那个上界：省下来的访存只有一部分变成了时间。

瓶颈落在矢量流水上：规约与指数都排在同一条流水里，而**融合能省掉的只是搬运，省不掉计算**。这与实验五是同一条判据，只是本实验把它推到了另一侧——**实验五的算子链访存受限，融合收益接近理论值；本实验的算子链在单核上受矢量流水限制，收益低于理论值。同一条判据，两种结果。**

**③ 沿行切分为什么值得单独一提**

v3 与 v4 是同一个核函数，差别只在启动的核数，**因此这一对的差值只能归因于并行度**。

沿行切分**没有核间合并**：每个核独占若干整行，算完直接写出，既不需要 workspace 两阶段
合并，也不需要原子累加。实验三的规约必须在核间合并，那一部分工作无法被并行摊薄；
实验三、实验五都用 $t(b) = I + S/b$ 拟合过，解出的截距正是这部分工作的度量。
**本实验不必再拟合一次**：切分方式已经决定了截距很小，v3 到 v4 的比值应当接近核数之比。

**扩展性的上限由能切出多少块决定，而不是由设备有多少核决定**——这条判据自实验三起反复
出现。本实验每核仍分得上百行，远未触到这个上限；走平要等到核数逼近行数。

**④ 三项判据的分工**

这是本实验的核心结论。第 11 节的第二张图集中体现了三项判据的分工：它们的失效位置并不相同，按幅度从小到大——

- 幅度较小时，三项都不失效；
- 幅度**尚未越过** $\ln(\text{FLT\_MAX})$ 时，就已经有判据失效：每个 $t_{ij}$ 都还是有限值（有限性通过），但分母是 $N$ 项之和、已经溢出为 `inf`，于是输出整体被压成 0，绝对误差与行和同时失效。**这正是第 1.2 节所说的分母比单项更早溢出。**
- 幅度越过 $\ln(\text{FLT\_MAX})$ 之后，最大的那些 $t_{ij}$ 本身成为 `inf`，`inf / inf` 产生 `nan`，有限性随之失效；
- **幅度最大的那几档，绝对误差反而不再失效**：真值最大的那些元素恰好就是变成 `nan` 的那些，它们被排除在误差统计之外（第 8.1 节）；留在统计里的元素真值本就接近 0，计算结果也是 0，逐元素比对因此测不出差别。

**只有不变量这一项，在整个失效区间内始终有效。** 绝对误差的表现尤其值得注意：**它在靠近边界处有效，在偏差最大的区间反而失效**。若只用这一条判据，会把一个完全错误的结果判为通过。

由此得到一条可推广的结论：**逐元素比对检查的是算得准不准，不变量检查的是算的还是不是这个东西。** 后者不需要参考实现，代价很小，却能捕捉前者结构性地看不见的失效。

**⑤ 手写实现与高阶 API 的差距从哪里来**

v4 与 v5 的核数、访存量、切分方式完全相同，**差别只在三趟由谁完成**，因此两者的比值直接反映实现效率。

高阶 API 在本实验的规格下明显快于手写版，两台设备上的倍数也接近。这一结果有据可查。官方在讨论指令执行延迟时给出过归约方案的排序：

> 根据单指令性能测试数据分析，WholeReduceSum 等归约指令的延迟时间约为 Add 指令的 2-5 倍。……通常来说，数据量较大、循环次数较多的场景，二分累加方案性能 > WholeReduceSum 单指令操作性能 > ReduceSum 接口性能。
> ——《Ascend C 算子开发指南》 指令执行延迟

**手写版每行要调用两次 `ReduceMax` / `ReduceSum`，而这一类接口恰好排在上面这个序列的最后。**

第二条依据来自第 7.6 节那张官方算法框图：**API 内部用 broadcast 把行标量摊成一个 datablock，再对整段做减法与除法，七个步骤全部在矢量单元上完成。** 手写版则要用 `GetValue` 把行标量取到标量流水，再作为立即数送回矢量指令——按第 2.1 节的官方说明，这意味着矢量流水要等标量操作结束，每行两次。**两条依据指向同一件事：高阶 API 的实现没有离开矢量流水，手写版每行离开两次。**

**这一对照的价值不在于谁更快。** 它给出的是一条可复用的判断：当手写实现里出现「整段归约 → 取标量 → 再作用回整段」这一模式时，**先查有没有对应的高阶 API；没有，再按官方的二分累加方案改写归约**（动手练习第 2 题）。高阶 API 另外还换来两样东西——换设备时的可移植性，以及写法的确定性：三趟的顺序与临时空间的复用都不再由开发者负责。

### 高阶 API 对主机侧的两项依赖

实验六第 12 节用 `Matmul` 高阶 API 做了一遍矩阵乘，本实验的 v5 用 `SoftMax` 高阶 API
做了一遍 Softmax。**两者都是在核直调工程里完成的，一个 `.asc` 文件、一条编译命令。**
官方对此有明确说法：谈到 Matmul 的系统 workspace 时，它把工程分成三类——

> 若算子工程不是自定义算子工程，也不是带有 HAVE_WORKSPACE 编译宏的 Kernel 直调算子工程，
> 框架不会自动设置 workspace……
> ——《Ascend C 算子开发指南》 Matmul 高阶 API 的 workspace 说明

**「带 HAVE_WORKSPACE 编译宏的 Kernel 直调算子工程」与自定义算子工程并列**，
因此「用了高阶 API 就必须上算子工程」这一说法并不成立。

两者的真正差别，在于**主机侧要为它准备什么**：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 主机侧要准备的东西 | <code>Matmul</code>（实验六第 12 节） | <code>SoftMax</code>（本实验 v5） |
| --- | --- | --- |
| 切分参数 | 必须由 Tiling 库算出。主机侧调用 <code>MultiCoreMatmulTiling</code> 解出 <code>TCubeTiling</code>，再经一块设备内存传入 | 两条路都可以：主机侧用 <code>SoftMaxTilingFunc</code> 算好传入，或传默认值由 API 按 shape 重算（<code>isCheckTiling</code>）|
| 临时空间 | 需要框架管理的<strong>系统 workspace</strong>。核直调工程靠 <code>HAVE_WORKSPACE</code> 编译宏取得 | 由开发者用 <code>TBuf</code> 自行申请，经 <code>sharedTmpBuffer</code> 入参传入 |
| 核内如何取用 | <code>REGIST_MATMUL_OBJ</code> 注册之后由 Matmul 对象管理 | 直接作为函数实参 |
| 主机侧代码量 | 两件准备工作，须写 Tiling 与 workspace 申请 | 可以为零，本版即是 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">主机侧要准备的东西</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;"><code>Matmul</code>（实验六第 12 节）</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;"><code>SoftMax</code>（本实验 v5）</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">切分参数</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">必须由 Tiling 库算出。主机侧调用 <code>MultiCoreMatmulTiling</code> 解出 <code>TCubeTiling</code>，再经一块设备内存传入</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">两条路都可以：主机侧用 <code>SoftMaxTilingFunc</code> 算好传入，或传默认值由 API 按 shape 重算（<code>isCheckTiling</code>）</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">临时空间</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">需要框架管理的<strong>系统 workspace</strong>。核直调工程靠 <code>HAVE_WORKSPACE</code> 编译宏取得</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">由开发者用 <code>TBuf</code> 自行申请，经 <code>sharedTmpBuffer</code> 入参传入</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">核内如何取用</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>REGIST_MATMUL_OBJ</code> 注册之后由 Matmul 对象管理</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">直接作为函数实参</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">主机侧代码量</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">两件准备工作，须写 Tiling 与 workspace 申请</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">可以为零，本版即是</td>
</tr>
</tbody>
</table>

官方在临时空间那一节把矩阵计算单列为例外（「除矩阵计算、HCCL 通信类、卷积计算等」），
正与上表第二行对应。

**由此得到一条可用的判断方法**：一个高阶 API 在核直调里好不好用，看的不是它是不是高阶 API，
而是它对主机侧的这两项依赖——切分参数从哪里来、临时空间由谁管理。依赖越少，
在核直调里用起来越省事；依赖越多，主机侧要写的准备代码越长，做成算子工程的理由也越充分。

### 其余几项的判读提示

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 现象 | 怎么读 |
| --- | --- |
| v1 在主规格下判定为 FAIL | 这是设计如此：主规格的输入幅度已越过 <code>float32</code> 的量程，v1 不做数值稳定化必然失效。它仍可用作计时基准，因为它与 v2 的算法只差一趟 |
| 行长超过第 2.3 节的上界 | 全融合不再可行。这不是实现缺陷，而是融合这一手法本身的适用范围：它要求被融合的几趟共享同一块驻留数据。出路是退回非融合，或改用在线算法（练习 5） |
| CPU 基准的算法与 v3 相同 | 加速比因此反映的是同一个算法在两种处理器上的差距，不含算法本身的差别。基准是单线程的，换成多线程后加速比会整体下降，但版本之间的相对关系不变 |
| 两台机器的耗时不同、结论相同 | 核函数耗时与设备型号、CANN 版本、运行时负载都有关。应当核对的是各条判据的先后与量级关系，而不是具体数值 |
| 行数改为不能被核数整除的值仍然通过 | 说明第 7.7 节的切分逻辑正确。这一项必须单独验证：主规格的行数恰好能被常用核数整除，掩盖了尾块的处理 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">现象</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">怎么读</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v1 在主规格下判定为 FAIL</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">这是设计如此：主规格的输入幅度已越过 <code>float32</code> 的量程，v1 不做数值稳定化必然失效。它仍可用作计时基准，因为它与 v2 的算法只差一趟</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">行长超过第 2.3 节的上界</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">全融合不再可行。这不是实现缺陷，而是融合这一手法本身的适用范围：它要求被融合的几趟共享同一块驻留数据。出路是退回非融合，或改用在线算法（练习 5）</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">CPU 基准的算法与 v3 相同</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">加速比因此反映的是同一个算法在两种处理器上的差距，不含算法本身的差别。基准是单线程的，换成多线程后加速比会整体下降，但版本之间的相对关系不变</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">两台机器的耗时不同、结论相同</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">核函数耗时与设备型号、CANN 版本、运行时负载都有关。应当核对的是各条判据的先后与量级关系，而不是具体数值</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">行数改为不能被核数整除的值仍然通过</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">说明第 7.7 节的切分逻辑正确。这一项必须单独验证：主规格的行数恰好能被常用核数整除，掩盖了尾块的处理</td>
</tr>
</tbody>
</table>

### 🎓 结论

Softmax 把本章前六个实验的结论集中用在了一个算子上：规约的写法来自实验三，量程的分析来自
实验四，融合的判据来自实验五，高阶 API 的用法来自实验六。**五个版本的每一处设计都能追溯到
其中某一条。**

而本实验自己新增的一条，是判据：**逐元素比对检查的是算得准不准，不变量检查的是算的还是不是
这个东西。** 前者需要参考实现，后者只需要输出本身；前者在结构性失效面前会失灵，后者不会。
一个算子交付之前，两者都要有。


## 14. 🔧 动手练习

1. **测出行长的上界。** 逐步增大 `LAB_COLS` 重新编译（例如 2048、4096、8192、16384），找出编译开始失败的转折点，与第 2.3 节表中最低的那个上界对照。两者若不相等，请说明还有哪些片上占用没有计入。

2. **按官方的二分累加方案改写行内求和。** 官方指出 `ReduceSum` 接口的性能低于单条归约指令，更低于二分累加方案。请改写 v3 的行内求和——先用 `Add` 反复对折，数据量降到一个 repeat 以内后再用一条归约指令收尾——测量它与 `ReduceSum` 接口的差距，并与 v5 对照，看这一项改写能补上多少差距。

3. **改变输入的分布。** 把 `GenData` 改成让各行的最大值相差若干个数量级（例如第 $i$ 行乘以 $10^{i \bmod 5}$）。v2、v3、v4 是否仍然通过校验？据此说明：减去**行**最大值而不是**全局**最大值，为什么是必要的。

4. **验证不变量判据的独立价值。** 在 v3 的核函数里故意让 `ReduceSum` 少算最后一个元素（把 `cols_` 改成 `cols_ - 8`）。三项判据分别是什么结果？哪一项先发现问题？

5. **【进阶】实现在线 Softmax。** 设已处理完前 $k$ 个元素，得到最大值 $m_k$ 与修正后的和 $s_k$。读入第 $k+1$ 个元素 $x$ 后：
   $$ m_{k+1} = \max(m_k,\, x), \qquad s_{k+1} = s_k \cdot e^{m_k - m_{k+1}} + e^{x - m_{k+1}} $$
   请先验证这两条递推式与三趟算法给出同样的结果，再据此实现一个分块版本，使片上占用与行长无关。测量它在长行上相对 v2 的表现。

6. **改变交给高阶 API 的形状。** v5 每次只交给 API 一行（`srcM = 1`）。改为一次交给 8 行（`srcM = 8`、`srcK = cols`），相应放大输入输出队列与 `API_TMP_BYTES`，重新测量。这一改动同时满足了 `isBasicBlock` 的两个条件（第 7.6 节），可以把该模板参数一并打开再测一次。变快了多少？据此说明：逐行调用把什么开销重复了多少次。

7. **【进阶】把切分参数改由主机侧算出。** 按官方给出的两步流程重做 v5：先用 `GetSoftMaxMinTmpSize` 与 `GetSoftMaxMaxTmpSize` 定出临时空间的上下界，在其中选一个值；再用 `SoftMaxTilingFunc` 按 shape 与该值算出 `SoftMaxTiling`，用一块 Global Memory 传到核函数。与 v5 比较耗时，据此回答：**`isCheckTiling` 让 API 每次调用都重算一遍切分，代价有多大？** 这一版的写法与实验六第 12.6 节的 `TCubeTiling` 是同一条路径，可以对照着写。


## 15. 🤔 思考题

- 第 1.2 节证明了减去任意常数 $c$ 都不改变结果。**既然如此，为什么一定要取行最大值？** 取一个固定的大常数（例如 88）是否可行？请分别从上溢与下溢两个方向说明。
- 本实验的第二趟同时产出 $t$ 与 $s$。**若把它拆成两个核函数——一趟算 $t$、一趟算 $s$——访存量会变成多少？** 这个拆分在什么情况下反而是合理的？
- v3 与 v4 是同一个核函数。**这意味着融合与多核是两个正交的改动吗？** 请找出一种情形，使得两者不再正交。
- 第 3.3 节的不变量是逐行求和为 1。**请为实验二的向量加法、实验三的规约、实验六的矩阵乘各设计一条不变量判据**，并说明哪一个算子的不变量最难设计、为什么。
- 在线 Softmax 用一趟完成了三趟的工作，**它是否违反了第 1.1 节关于分母依赖整行的论断？** 请说明它付出了什么代价来换取这一点。
- 若输入的数据类型改为 `half`，`exp` 的上溢边界会从 88.7 降到多少？**在这种情况下，减去行最大值是否仍然足够？** 若不够，还要补充什么处理？
- 本实验的 CPU 基准是单线程的。**若改为多线程，第 10 节的加速比会怎样变化，哪些结论会因此失效？** 哪些结论不受影响？
- 第 13 节 ④ 指出绝对误差判据在输出接近 0 时会失去分辨力。**这是否意味着应当改用相对误差？** 请说明相对误差在 Softmax 上会遇到什么新问题。
- 第 2.3 节的表里，v5 的行长上界是六个核函数中最低的。**这是 `SoftMax` 高阶 API 本身的性质，还是本实验给它预留临时空间的方式造成的？** 若改按官方的 `GetSoftMaxMinTmpSize` 取最小值，结论会变吗？
- 第 13 节把高阶 API 对主机侧的依赖归为两项：切分参数从哪里来、临时空间由谁管理。**一个需要跨核通信的高阶 API（例如全局规约）在这两项上会是什么情形？** 它在核直调里会比 `SoftMax` 更难用还是更好用？说明理由。

## 16. 📌 本实验小结

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 概念 | 要点 |
| --- | --- |
| 三趟依赖 | 分母依赖整行，因此 Softmax 不是逐元素算子；加上稳定化处理共三趟 |
| 数值稳定化 | 减去行最大值不改变数学结果，使指数的自变量恒不大于 0，消除上溢 |
| 上溢与下溢 | 减最大值消除上溢；下溢为 0 是正确的，真值本就不可表示 |
| 逐行规约 | `ReduceMax` 与 `ReduceSum` 与实验三同一个接口，临时空间按同一条式子核算 |
| 行标量的写出 | 每行一个标量，逐行写出违反 32 字节粒度，必须在片上攒够再整体搬出 |
| 标量与整行 | `Adds` / `Muls` 的第三个参数本就是标量，无须广播接口 |
| 乘法代替除法 | 先求标量倒数再 `Muls`，每行只做一次除法 |
| 全融合 | 整行驻留片上，三趟之间不落回 Global Memory，访存量由 5 份降到 2 份 |
| 行长上界 | 由片上容量定出；超过上界时全融合不再适用 |
| 沿行切分 | 切分单位是行，行首地址天然对齐（要求 $N$ 是 8 的倍数）；行数不整除时前 rem 个核各多一行 |
| 多核扩展性 | 沿行切分无核间合并，每个核独占若干整行、算完直接写出；与核数无关的那部分工作因此远小于实验三的规约 |
| 编译期裁剪版本差异 | v1 与 v2 共用一份代码，由模板参数与 `if constexpr` 在编译期选择是否减最大值 |
| **三项判据** | 绝对误差、有限性、不变量；三者失效的位置不同，缺一不可 |
| **不变量的价值** | 不需要参考实现，能捕捉逐元素比对结构性看不见的失效 |
| 高阶 API 的用法 | `AscendC::SoftMax(dst, src, sharedTmp, tiling, shapeInfo)`；临时空间由开发者给，`tiling` 置零时 API 按 shape 自行推导 |
| 高阶 API 与算子工程 | 两个条件决定是否必须上工程：切分参数能否在核内推导、是否需要框架管理的 workspace。`Matmul` 两条都占，`SoftMax` 都不占 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">概念</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">要点</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">三趟依赖</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">分母依赖整行，因此 Softmax 不是逐元素算子；加上稳定化处理共三趟</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">数值稳定化</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">减去行最大值不改变数学结果，使指数的自变量恒不大于 0，消除上溢</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">上溢与下溢</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">减最大值消除上溢；下溢为 0 是正确的，真值本就不可表示</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">逐行规约</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>ReduceMax</code> 与 <code>ReduceSum</code> 与实验三同一个接口，临时空间按同一条式子核算</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">行标量的写出</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">每行一个标量，逐行写出违反 32 字节粒度，必须在片上攒够再整体搬出</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">标量与整行</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>Adds</code> / <code>Muls</code> 的第三个参数本就是标量，无须广播接口</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">乘法代替除法</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">先求标量倒数再 <code>Muls</code>，每行只做一次除法</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">全融合</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">整行驻留片上，三趟之间不落回 Global Memory，访存量由 5 份降到 2 份</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">行长上界</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">由片上容量定出；超过上界时全融合不再适用</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">沿行切分</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">切分单位是行，行首地址天然对齐（要求 N 是 8 的倍数）；行数不整除时前 rem 个核各多一行</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">多核扩展性</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">沿行切分无核间合并，每个核独占若干整行、算完直接写出；与核数无关的那部分工作因此远小于实验三的规约</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">编译期裁剪版本差异</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v1 与 v2 共用一份代码，由模板参数与 <code>if constexpr</code> 在编译期选择是否减最大值</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>三项判据</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">绝对误差、有限性、不变量；三者失效的位置不同，缺一不可</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>不变量的价值</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">不需要参考实现，能捕捉逐元素比对结构性看不见的失效</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">高阶 API 的用法</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>AscendC::SoftMax(dst, src, sharedTmp, tiling, shapeInfo)</code>；临时空间由开发者给，<code>tiling</code> 置零时 API 按 shape 自行推导</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">高阶 API 与算子工程</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">两个条件决定是否必须上工程：切分参数能否在核内推导、是否需要框架管理的 workspace。<code>Matmul</code> 两条都占，<code>SoftMax</code> 都不占</td>
</tr>
</tbody>
</table>

### 一条贯穿本章的原则

> **算子的正确性不是与性能并列的一项指标，而是性能得以被讨论的前提。** 本实验的 v1 比 v2 快，但它算错了；一个算错的实现，快慢没有意义。本章前六个实验反复在做的事——核算访存、分析量程、设计判据——都是为了让性能上的结论站得住。

### 第六章总结

七个实验走完了一条完整的路径：

1. **实验一至实验二**：核函数的结构、搬运与计算的编排、分块与队列；
2. **实验三**：规约——第一个不满足逐元素性质的算子，两级规约与核间合并；
3. **实验四**：量程与精度——`half` 的表示范围，两类失效的区分；
4. **实验五**：融合——访存量而非计算量决定收益；
5. **实验六**：矩阵乘——从矢量单元到 Cube，以及高阶 API 的引入；
6. **实验七**：把上述结论用在同一个算子上，并用高阶 API 再实现一遍作为对照。

**这条路径的次序不是随意的。** 前面每一个实验都在为后面的实验准备判据：访存量核算准备了融合的上界，量程分析准备了数值稳定化的依据，规约的两级结构准备了逐行规约的写法。到本实验时，五个版本的每一处设计都能追溯到前面某一个实验的结论——**这正是综合实验的含义。**

### 与后续内容的衔接

➡️ 第七章将从单个算子转向**算子之间**：计算图的组织、内存的复用、以及多个算子协同时出现的新问题。本章反复用到的访存量决定收益这条判据，在图这一层会以另一种形式重新出现。